# NLP HW3 — CFEVER 中文事實查核（最終版）

**學號：115062556　　Kaggle username：17568160**　（Report 在本 notebook 最後一個 cell）

### Generative AI 使用聲明
本作業的整體架構設計與程式碼由 Anthropic **Claude（Sonnet 5.5：v0–v8；Opus 5.5：v9）** 協助撰寫，內容包含：
混合式頁面檢索與學習式排序、句子 cross-encoder 重排與證據句選擇器、多個分類器的集成、Kaggle GPU 上的訓練程式、提交檔產生與統計分析。
以上程式由本人在本機（RTX 3050 4GB）與本人的 Kaggle Notebook GPU（私人 notebook / dataset）實際執行、檢查並依結果調整；實驗數據皆為本人實際跑出。
使用的預訓練模型（每個 < 1B 參數、未呼叫任何外部 API）：`BAAI/bge-small-zh-v1.5`、`hfl/chinese-macbert-base`、`hfl/chinese-roberta-wwm-ext`、
`IDEA-CCNL/Erlangshen-Roberta-110M-NLI`、`IDEA-CCNL/Erlangshen-Roberta-330M-NLI`、`Qwen/Qwen3-0.6B-Base`（LoRA）、
`joeddav/xlm-roberta-large-xnli`、`MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7`。
其中 Erlangshen NLI、XLM-R XNLI、mDeBERTa XNLI 為已在外部 NLI 資料微調過的 checkpoint，已向助教確認此類 checkpoint 可用（參數量 < 1B）；訓練只用 cfever 的 train / dev / wiki。
參考資料：作業提供的 sample code（資料讀取、評分函式、提交格式）、CFEVER 論文與資料集說明（https://github.com/IKMLab/CFEVER-data）、Hugging Face 各模型的 model card。

### Pipeline
1. **Document Retrieval**：標題子字串比對 + BM25（字元 bigram）+ 稠密檢索（bge-small-zh）→ 用 HistGradientBoosting 學習式排序，取前 6 頁。
2. **Evidence Retrieval**：對候選頁面的句子，用微調過的 cross-encoder（MacBERT）打分，再用證據句選擇器（HistGradientBoosting）重排，取前 5 句。
   （第二跳、第二個重排器都實驗過，沒有明確好處，沒有採用，見第 5 節與 Report。）
3. **Classification**：12 個分類器（MacBERT、RoBERTa、Erlangshen-110M / 330M-NLI、Qwen3-0.6B + LoRA、XLM-R-large-XNLI、mDeBERTa-v3-XNLI；
   其中 6 個在輸入中附上 2～3 句「兄弟 claim」，見第 9 節）以 claim + 前 4 句證據分三類，log-softmax 後等權平均，再對 NOT ENOUGH INFO 加偏差
   （偏差的選法：讓 test 的 NEI 預測比例對齊 v7 在偏差 1.75 / 2.0 時的比例，見第 9 節）。

與 sample code 的主要差異：sample code 只索引「訓練集涉及的頁面」，但 dev 的證據頁面只有約 11% 出現在其中；
本版改用**完整 wiki（約 119 萬頁）**，並以廉價檢索縮小候選後才使用 BERT 類模型。

**最終提交的兩份檔案**（Kaggle 上選這兩份計分）：`submission_v9_m12_share1.75.csv`（public 0.889）與 `submission_v9_m12_share2.0.csv`（public 0.888），
由第 9 節 9e 的 cell 產生（輸出資料夾 `output/<VERSION>/`）。第 7、8 節是 v7 / v8 的結果（v7 的 6 成員集成，public 0.881），保留作為對照與消融。

**執行方式**：先把 `QUICK_TEST = True` 跑一遍（用 `wiki_sample.jsonl`，約十分鐘）確認環境與流程無誤，再改成 `False` 做完整執行。
每個階段的結果都會存到 `output/cache/`，中途中斷後重跑會自動略過已完成的階段。

In [1]:
# ===== 1. 套件與參數 =====
# （Generative AI 使用聲明：本 notebook 程式碼由 Claude（Sonnet 5.5 / Opus 5.5）協助撰寫，詳見上方說明與 Report Q5。）
import os, re, json, math, time, random, pickle, gc, functools, collections
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import (AutoTokenizer, AutoModel, AutoModelForSequenceClassification,
                          get_linear_schedule_with_warmup)
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold

# ---------- 最重要的開關 ----------
QUICK_TEST = False        # True：用 wiki_sample.jsonl 與少量資料，十分鐘內跑完整條流程（先用它檢查環境）
FORCE_RETRAIN = False     # True：忽略已存的模型快取，重新訓練（包含重排器，約 40 分鐘）
FORCE_RETRAIN_CLASSIFIER = False   # True：只重新訓練分類器
SEED = 2026

# ---------- 檢索參數 ----------
LEAD_RAW_CHARS, LEAD_CHARS = 400, 110   # 頁面「摘要」= 標題 + 前 110 字（BM25 與稠密檢索使用）
BM25_FEATURES, BM25_K1, BM25_B, BM25_MAX_DF = 2 ** 21, 1.2, 0.75, 0.05
K_BM25, K_DENSE = 50, 50                # BM25 / 稠密檢索各取前幾名
CAND_BM25, CAND_DENSE = 20, 20          # 其中前幾名進入候選集
TITLE_MIN, TITLE_MAX = 2, 24            # 標題比對的長度範圍（單字標題噪音太大，直接略過）
PAGES_PER_TITLE, MAX_TITLE_CANDS = 6, 40
USE_DENSE = True
DENSE_MODEL = "BAAI/bge-small-zh-v1.5"
DENSE_MAX_LEN, DENSE_BS = 96, 256

# ---------- 句子檢索 ----------
PAGE_TOPK = 6                           # 取排序後前幾頁的句子當候選
PER_PAGE_MAX_SID = 60                   # 每頁只看前 60 句（gold 有 95% 在 50 句內）
RERANK_POOL = 32                        # 每個 claim 送進 cross-encoder 的句子數
USE_HOP2 = True                         # 是否沿超連結做第二跳
HOP_SRC, HOP_MAX_PAGES, HOP_PER_PAGE, HOP_POOL = 2, 6, 4, 16
RERANK_MODEL = "hfl/chinese-macbert-base"
RERANK_MAX_LEN, RERANK_NEG = 128, 5
RERANK_EPOCHS, RERANK_LR, RERANK_BS, RERANK_ACCUM = 1, 3e-5, 16, 2
RERANK_TRAIN_CLAIMS = None              # None = 使用全部 supports/refutes 訓練題；顯存/時間不夠可設成 8000

# ---------- 分類器 ----------
CLS_MODEL = "hfl/chinese-macbert-base"  # 若載入失敗可改成 "google-bert/bert-base-chinese"
CLS_CTX_K, CLS_MAX_LEN = 4, 256
CLS_EPOCHS, CLS_LR, CLS_BS, CLS_ACCUM = 2, 2e-5, 8, 4
# 分類器集成（v2）：多個不同骨幹/種子的分類器，logits（log-softmax）取平均。每個成員訓練完都存檔，之後重跑會直接載入。
#   - macbert_base 是 v1 已訓練好的分類器（classifier_v2），不會重訓。
#   - 想加成員就把 tag 加進 ACTIVE_MEMBERS：先跑 roberta_base（約 35 分鐘），確認有進步再加 macbert_large（約 1.5~2 小時）。
CLS_MEMBERS = [
    dict(tag="macbert_base", model="hfl/chinese-macbert-base", path="classifier_v2", max_len=CLS_MAX_LEN),
    dict(tag="roberta_base", model="hfl/chinese-roberta-wwm-ext", lr=2e-5, epochs=2, bs=8, accum=4, seed=11),
    # 先在中文 NLI 資料（CMNLI/OCNLI 等，約 100 萬筆）微調過的 checkpoint（IDEA-CCNL Erlangshen，Apache-2.0，皆 < 1B）。
    # NLI 的 (前提, 假設) 對應這裡的 (證據, claim)，所以 swap=True 把證據放前面；NLI 標籤順序是 矛盾/中立/蘊含，
    # head_perm 把分類頭重排成 supports/refutes/NEI = 蘊含/矛盾/中立，讓微調從「已經懂得推論」的起點開始。
    dict(tag="erlangshen_110m", model="IDEA-CCNL/Erlangshen-Roberta-110M-NLI", lr=2e-5, epochs=2, bs=8, accum=4, seed=21,
         swap=True, head_perm=[2, 0, 1]),
    # 330M（large）：4GB 顯存放不下完整 AdamW，改用 Adafactor + 凍結 embedding 與前 8 層（OOM 時自動多凍結 4 層再重試）。
    # 注意：不要用 Erlangshen-MegatronBert-1.3B-NLI，它超過助教規定的 1B 上限。
    dict(tag="erlangshen_330m", model="IDEA-CCNL/Erlangshen-Roberta-330M-NLI", lr=1.5e-5, epochs=2, bs=4, accum=8, seed=22,
         swap=True, head_perm=[2, 0, 1], freeze=8, optim="adafactor", infer_bs=16),
    # 同一個 330M 架構的第二顆種子（約 2 小時）。seed 會同時改變分類頭初始化、dropout、資料順序與訓練資料的建立順序，
    # 兩顆模型的錯誤因此不完全重疊，平均後變異數下降 —— 這是「真的」泛化提升（可在 dev 驗證），不是照 public 挑出來的。
    dict(tag="erlangshen_330m_b", model="IDEA-CCNL/Erlangshen-Roberta-330M-NLI", lr=1.5e-5, epochs=2, bs=4, accum=8, seed=42,
         swap=True, head_perm=[2, 0, 1], freeze=8, optim="adafactor", infer_bs=16),
    # 小型 LLM（Qwen3-0.6B-Base，約 0.6B 參數 < 1B）+ LoRA。需要 peft（pip install peft）。
    # 先跑 qwen06_pilot（只用 6000 筆訓練資料、1 epoch）：dev accuracy 若已達 ~0.72 以上就值得跑完整版 qwen06。
    dict(tag="qwen06_pilot", kind="llm", model="Qwen/Qwen3-0.6B-Base", lr=1e-4, epochs=1, bs=4, accum=4, seed=31, n_train=6000,
         max_len=384, infer_bs=16),
    dict(tag="qwen06", kind="llm", model="Qwen/Qwen3-0.6B-Base", lr=1e-4, epochs=2, bs=4, accum=4, seed=32, max_len=384, infer_bs=16),   # v9 使用（見第 9 節）
]
ACTIVE_MEMBERS = ["macbert_base", "roberta_base", "erlangshen_110m", "erlangshen_330m", "erlangshen_330m_b", "qwen06_pilot"]
NEW_MEMBER = "erlangshen_330m_b"        # 下面會做「有/沒有這個成員」的配對比較，判斷提升是不是雜訊
PRIOR_TARGET = (1 / 3, 1 / 3, 1 / 3)    # 僅用於「舊的先驗修正」消融；正式預測改用 dev 校準（見分類評估 cell）
CLS_TRAIN_SR_CLAIMS = None              # None = 所有 supports/refutes 訓練題都用重排器重排一次，當作分類訓練證據（約 25 分鐘）
CLS_SHUFFLE_P = 0.3                     # 訓練時有 30% 的機率把證據順序打亂，避免模型依賴「標準證據排第一」
HOP_PENALTIES = (0.0, 1.0, 2.0, 3.0, 5.0, None)   # 第二跳候選的扣分；None = 不使用第二跳。依 dev 選擇

# ---------- 共用 ----------
GRAD_CKPT = True                        # gradient checkpointing：省顯存（約慢 30%）。顯存夠可設 False
INFER_BS = 64
N_TRAIN_LIMIT = N_DEV_LIMIT = N_TEST_LIMIT = None

if QUICK_TEST:
    N_TRAIN_LIMIT, N_DEV_LIMIT, N_TEST_LIMIT = 800, 200, 100
    RERANK_TRAIN_CLAIMS = 400
    RERANK_EPOCHS = CLS_EPOCHS = 1

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"
print("裝置：", device, torch.cuda.get_device_name(0) if USE_AMP else "（沒有 GPU，會非常慢）")

# ---------- 路徑 ----------
IS_KAGGLE = Path("/kaggle/input").exists()
if IS_KAGGLE:
    DATA_DIR, OUTPUT_ROOT = Path("/kaggle/input"), Path("/kaggle/working")
else:
    DATA_DIR = next((p for p in (Path("dataset"), Path(".")) if (p / "train.jsonl").is_file()), Path("dataset"))
    OUTPUT_ROOT = Path("output")
VERSION = "final"                                  # 提交檔與 metrics.json 放進 output/<VERSION>/，每個版本各一個資料夾
OUTPUT_DIR = OUTPUT_ROOT / VERSION
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def find_file(name):
    direct = DATA_DIR / name
    if direct.is_file():
        return direct
    hits = sorted(p for p in DATA_DIR.rglob(name) if p.is_file())
    if not hits:
        raise FileNotFoundError(f"找不到 {name}；目前搜尋目錄：{DATA_DIR.resolve()}")
    return hits[0]


TRAIN_PATH, DEV_PATH, TEST_PATH = find_file("train.jsonl"), find_file("dev.jsonl"), find_file("test.jsonl")
WIKI_PATH = find_file("wiki_sample.jsonl" if QUICK_TEST else "wiki_pages.jsonl")
CACHE_DIR = OUTPUT_ROOT / ("cache_quick" if QUICK_TEST else "cache")   # 快取不跟著版本走（否則所有模型與推論都要重算）
CACHE_DIR.mkdir(parents=True, exist_ok=True)
print("資料夾：", DATA_DIR.resolve(), "| wiki：", WIKI_PATH.name, "| 輸出：", OUTPUT_DIR.resolve())

/home/yiting/venvs/hw3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


裝置： cuda NVIDIA GeForce RTX 3050 Laptop GPU
資料夾： /home/yiting/NLP-Assignment3/dataset | wiki： wiki_pages.jsonl | 輸出： /home/yiting/NLP-Assignment3/output/final


In [2]:
# ===== 2. 共用小工具 =====
_SPACE = re.compile(r"(?<=[^\x00-\x7f])\s+|\s+(?=[^\x00-\x7f])")   # 只去掉與中文字（非 ASCII）相鄰的空白
_PAREN = re.compile(r"_\(.*\)$")


def norm(s):
    """wiki 文字是斷詞過的（『數學 ， 是研究…』），claim 沒有空格；統一去掉與中文相鄰的空白。"""
    return _SPACE.sub("", s).strip()


def title_base(t):
    """'愛爾蘭_(消歧義)' -> '愛爾蘭'（用來和 claim 做子字串比對）"""
    return _PAREN.sub("", t).replace("_", " ")


def bigrams(s):
    if len(s) < 2:
        return {s} if s else set()
    return {s[i:i + 2] for i in range(len(s) - 1)}


def read_jsonl(path):
    with open(path, encoding="utf-8-sig") as f:
        return [json.loads(l) for l in f if l.strip()]


def free_gpu():
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()


def amp_ctx():
    return torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP)


def cached(path, build_fn, force=False):
    """有快取就讀，沒有就執行 build_fn 並存檔。"""
    path = Path(path)
    if path.exists() and not force:
        with open(path, "rb") as f:
            return pickle.load(f)
    obj = build_fn()
    with open(path, "wb") as f:
        pickle.dump(obj, f, protocol=4)
    return obj


LABEL2ID = {"SUPPORTS": 0, "REFUTES": 1, "NOT ENOUGH INFO": 2}
LABEL_OUT = ["supports", "refutes", "NOT ENOUGH INFO"]   # 與 sample code 的輸出格式一致


def parse_gold(row):
    """回傳 [[(page_id, sentence_id), ...], ...]；NEI 沒有證據，回傳 []。"""
    if row["label"].upper() == "NOT ENOUGH INFO":
        return []
    groups = []
    for g in row["evidence"]:
        items = [(it[2], int(it[3])) for it in g if it[2] is not None and it[3] is not None]
        if items:
            groups.append(items)
    return groups

## 3. 讀取資料與完整 Wikipedia 索引

單次掃描 `wiki_pages.jsonl`（約 1GB）：記下每頁標題、檔案位移（之後需要句子時才回頭讀）、頁面長度，以及「標題 + 開頭」摘要。
之後所有檢索都建立在這份摘要上，所以不需要把 740 萬個句子放進記憶體。

In [3]:
def scan_wiki(path):
    titles, offsets, plen, leads = [], [], [], []
    pos = 0
    with open(path, "rb") as f:
        for raw in tqdm(f, desc="掃描 wiki", unit="頁", mininterval=5):
            n = len(raw)
            if raw.strip():
                page = json.loads(raw)
                t = page["id"]
                text = page.get("text", "") or ""
                titles.append(t)
                offsets.append(pos)
                plen.append(len(text))
                leads.append(title_base(t) + "：" + norm(text[:LEAD_RAW_CHARS])[:LEAD_CHARS])
            pos += n
    return dict(titles=titles, offsets=np.array(offsets, dtype=np.int64), plen=np.array(plen, dtype=np.int32), leads=leads)


_sig = f"{WIKI_PATH.stat().st_size}_{LEAD_CHARS}"
wiki = cached(CACHE_DIR / f"wiki_meta_{_sig}.pkl", lambda: scan_wiki(WIKI_PATH))
titles, offsets, plen, leads = wiki["titles"], wiki["offsets"], wiki["plen"], wiki["leads"]
NPAGES = len(titles)
title2idx = {t: i for i, t in enumerate(titles)}
base_titles = [title_base(t) for t in titles]
base2idx = defaultdict(list)           # '愛爾蘭' -> [頁面索引...]；標題本身（沒有括號）排最前面
for i, (t, b) in enumerate(zip(titles, base_titles)):
    if len(b) >= TITLE_MIN:
        if t.replace("_", " ") == b:
            base2idx[b].insert(0, i)
        else:
            base2idx[b].append(i)
print("頁面數：", NPAGES, "| 可比對的標題數：", len(base2idx))

WIKI_FH = open(WIKI_PATH, "rb")


@functools.lru_cache(maxsize=80000)
def get_sentences(pidx):
    """回傳 ((sentence_id, 去空白文字, 超連結目標頁面tuple), ...)；略過空句。"""
    WIKI_FH.seek(int(offsets[pidx]))
    page = json.loads(WIKI_FH.readline())
    out = []
    for line in page.get("lines", "").split("\n"):
        parts = line.split("\t")
        if len(parts) < 2 or not parts[0].isdigit():
            continue
        text = norm(parts[1])
        if text:
            out.append((int(parts[0]), text, tuple(x for x in parts[3::2] if x)))   # parts[2::2] 是錨文字，[3::2] 是目標頁面
    return tuple(out)


def disp(pidx):
    return titles[pidx].replace("_", " ")


def passage(pidx, text):
    return f"{disp(pidx)}：{text}"


print(get_sentences(0)[:2])

頁面數： 1187751 | 可比對的標題數： 1078608
((0, '數學，是研究數量、結構以及空間等概念及其變化的一門學科，屬於形式科學的一種。', ()), (2, '基礎數學的知識與運用是生活中不可或缺的一環。', ()))


In [4]:
# ----- 讀取 claim，建立 train / dev / test -----
def make_split(rows, name, labeled=True):
    s = dict(name=name, id=[r["id"] for r in rows], claim=[norm(r["claim"]) for r in rows])
    if labeled:
        s["label"] = np.array([LABEL2ID[r["label"].upper()] for r in rows])
        s["gold"] = [parse_gold(r) for r in rows]
    return s


def covered(groups):
    return any(all(t in title2idx for t, _ in g) for g in groups)


rng = random.Random(SEED)
train_rows, dev_rows, test_rows = read_jsonl(TRAIN_PATH), read_jsonl(DEV_PATH), read_jsonl(TEST_PATH)
if QUICK_TEST:   # 抽樣 wiki 只含部分頁面：只保留證據頁面都在抽樣內的題目
    ok = lambda r: r["label"].upper() == "NOT ENOUGH INFO" or covered(parse_gold(r))
    train_rows, dev_rows = [r for r in train_rows if ok(r)], [r for r in dev_rows if ok(r)]
for rows, lim in ((train_rows, N_TRAIN_LIMIT), (dev_rows, N_DEV_LIMIT), (test_rows, N_TEST_LIMIT)):
    if lim is not None and len(rows) > lim:
        rows[:] = rng.sample(rows, lim)

train = make_split(train_rows, "train")
dev = make_split(dev_rows, "dev")
test = make_split(test_rows, "test", labeled=False)
for s in (train, dev, test):
    print(s["name"], len(s["claim"]), (np.bincount(s["label"], minlength=3).tolist() if "label" in s else ""))
missing = sum(1 for s in (train, dev) for gs in s["gold"] for g in gs for t, _ in g if t not in title2idx)
print("標準答案頁面不在 wiki 的數量（完整模式應為 0；QUICK 模式因 wiki 為抽樣，可能不為 0）：", missing)

train 24012 [11085, 7113, 5814]
dev 3000 [1000, 1000, 1000]
test 6000 
標準答案頁面不在 wiki 的數量（完整模式應為 0；QUICK 模式因 wiki 為抽樣，可能不為 0）： 0


## 4. 第一階段：Document Retrieval

三種廉價訊號，各有所長：
- **標題比對**：claim 裡出現的 wiki 標題（dev 約 80% 的證據頁面標題會原封不動出現在 claim 中）。
  單字標題每題平均出現 13 個、幾乎都是噪音，所以只比對長度 ≥ 2 的標題。
- **BM25**（字元 bigram）：處理 claim 沒有直接寫出標題的情況；索引的是「標題 + 開頭 110 字」。
- **稠密檢索**（bge-small-zh）：處理改寫、同義詞。

三者取聯集當候選頁面，再用 **HistGradientBoosting** 學習怎麼排序（特徵包含標題長度、位置、是否被更長標題包含、
BM25 / 稠密檢索的分數與名次等）。標籤是「該頁是否為標準證據頁面」，只用訓練集，且訓練集的排序分數用 3 折交叉預測，避免過度樂觀。

In [5]:
# ----- BM25（字元 bigram，hashing）-----
HV = HashingVectorizer(analyzer="char", ngram_range=(2, 2), n_features=BM25_FEATURES, alternate_sign=False,
                       norm=None, lowercase=True, dtype=np.float32)


def build_bm25():
    mats = [HV.transform(leads[s:s + 100000]) for s in tqdm(range(0, NPAGES, 100000), desc="BM25 建索引")]
    X = sp.vstack(mats, format="csr")
    del mats
    df = np.bincount(X.indices, minlength=BM25_FEATURES)
    idf = np.log((NPAGES - df + 0.5) / (df + 0.5) + 1).astype(np.float32)
    idf[df > BM25_MAX_DF * NPAGES] = 0.0          # 幾乎每頁都有的 bigram（停用詞）直接忽略
    dl = np.asarray(X.sum(axis=1)).ravel().astype(np.float32)
    dl_rep = np.repeat(dl, np.diff(X.indptr))
    X.data = X.data * (BM25_K1 + 1) / (X.data + BM25_K1 * (1 - BM25_B + BM25_B * dl_rep / max(dl.mean(), 1e-6)))
    del dl_rep
    return X.tocsc(), idf


_bm25_path = CACHE_DIR / f"bm25_{_sig}.npz"
if _bm25_path.exists():
    BM25_X, BM25_IDF = sp.load_npz(_bm25_path), np.load(CACHE_DIR / f"bm25_idf_{_sig}.npy")
else:
    BM25_X, BM25_IDF = build_bm25()
    sp.save_npz(_bm25_path, BM25_X, compressed=False)
    np.save(CACHE_DIR / f"bm25_idf_{_sig}.npy", BM25_IDF)
BM25_PTR, BM25_IND, BM25_DAT = BM25_X.indptr, BM25_X.indices, BM25_X.data
print("BM25 索引：", BM25_X.shape, "非零元素：", BM25_X.nnz)


def bm25_scores(claim):
    q = HV.transform([claim])
    scores = np.zeros(NPAGES, dtype=np.float32)
    for c in q.indices:
        w = BM25_IDF[c]
        if w > 0:
            s, e = BM25_PTR[c], BM25_PTR[c + 1]
            scores[BM25_IND[s:e]] += w * BM25_DAT[s:e]
    return scores


def topk_idx(scores, k):
    k = min(k, len(scores))
    top = np.argpartition(-scores, k - 1)[:k]
    return top[np.argsort(-scores[top], kind="stable")]

BM25 索引： (1187751, 2097152) 非零元素： 84250765


In [6]:
# ----- 稠密檢索（bge-small-zh）-----
def load_dense_model():
    tok = AutoTokenizer.from_pretrained(DENSE_MODEL)
    model = AutoModel.from_pretrained(DENSE_MODEL).to(device).eval()
    if device.type == "cuda":
        model = model.half()
    return tok, model


@torch.no_grad()
def encode_dense(texts, tok, model, desc="編碼"):
    """CLS 向量 + L2 正規化，回傳 float16 numpy。依長度排序後分批，可省很多時間。"""
    order = np.argsort([len(t) for t in texts], kind="stable")
    out = np.zeros((len(texts), model.config.hidden_size), dtype=np.float16)
    for s in tqdm(range(0, len(texts), DENSE_BS), desc=desc, mininterval=5):
        ids = order[s:s + DENSE_BS]
        enc = tok([texts[i] for i in ids], padding=True, truncation=True, max_length=DENSE_MAX_LEN, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        h = model(**enc).last_hidden_state[:, 0]
        out[ids] = F.normalize(h.float(), dim=-1).cpu().numpy().astype(np.float16)
    return out


@torch.no_grad()
def dense_topk(Q, P, k, chunk=200000, qbatch=512):
    """Q:[n,d]、P:[N,d]（皆 float16 numpy）。分塊上 GPU 算內積並合併 top-k，避免一次吃光 4GB 顯存。"""
    n, k = len(Q), min(k, len(P))
    dt = torch.float16 if device.type == "cuda" else torch.float32
    Qg = torch.from_numpy(np.ascontiguousarray(Q)).to(device=device, dtype=dt)
    best_v = torch.full((n, k), -2.0, device=device)
    best_i = torch.zeros((n, k), dtype=torch.long, device=device)
    for ps in tqdm(range(0, len(P), chunk), desc="稠密檢索", mininterval=5):
        Pc = torch.from_numpy(np.array(P[ps:ps + chunk])).to(device=device, dtype=dt)
        for qs in range(0, n, qbatch):
            v, i = (Qg[qs:qs + qbatch] @ Pc.T).topk(min(k, Pc.shape[0]), dim=1)
            cv = torch.cat([best_v[qs:qs + qbatch], v.float()], 1)
            ci = torch.cat([best_i[qs:qs + qbatch], i + ps], 1)
            tv, ti = cv.topk(k, dim=1)
            best_v[qs:qs + qbatch], best_i[qs:qs + qbatch] = tv, ci.gather(1, ti)
    return best_i.cpu().numpy(), best_v.cpu().numpy()


_DENSE = {}


def dense_state():
    """第一次需要稠密檢索時才載入模型；頁面向量只算一次並存檔（約 10~20 分鐘）。"""
    if not _DENSE:
        tok, mdl = load_dense_model()
        path = CACHE_DIR / f"dense_pages_{_sig}_{DENSE_MODEL.split('/')[-1]}.npy"
        if not path.exists():
            np.save(path, encode_dense(leads, tok, mdl, desc="編碼全部頁面"))
        _DENSE.update(tok=tok, mdl=mdl, P=np.load(path, mmap_mode="r"))
        print("頁面向量：", _DENSE["P"].shape)
    return _DENSE

In [7]:
# ----- 標題比對 + 候選頁面 + 特徵 -----
FEATS = ["tm", "tlen", "tfrac", "tpos", "n_tm", "tm_rank", "nested", "paren", "same_base", "digit", "ptlen",
         "bm25", "bm25_rel", "bm25_rank", "dense", "dense_rel", "dense_rank", "plen", "lead_ov"]
FI = {f: i for i, f in enumerate(FEATS)}


def title_matches(claim):
    """claim 中所有以子字串形式出現的 wiki 標題（已去括號、長度 >= 2）。"""
    L, found = len(claim), {}
    for i in range(L):
        for j in range(i + TITLE_MIN, min(L, i + TITLE_MAX) + 1):
            sub = claim[i:j]
            if sub in base2idx and sub not in found:
                found[sub] = (i, j)
    return [(b, s, e) for b, (s, e) in found.items()]


def gen_features(claims, Q, d_idx, d_val, desc):
    rows_all, pidx_all, counts = [], [], []
    for qi, claim in enumerate(tqdm(claims, desc=desc, mininterval=5)):
        sc = bm25_scores(claim)
        top = topk_idx(sc, K_BM25)
        b_rank = {int(i): r for r, i in enumerate(top)}
        bmax = float(sc[top[0]]) if len(top) else 0.0
        if USE_DENSE:
            DENSE_P = dense_state()["P"]
            dq, di, dv = Q[qi].astype(np.float32), d_idx[qi], d_val[qi]
            d_rank = {int(i): r for r, i in enumerate(di)}
            dmax = float(dv[0])
        else:
            di, d_rank, dmax = np.array([], dtype=np.int64), {}, 0.0
        tms = sorted(title_matches(claim), key=lambda x: -(x[2] - x[1]))
        n_tm = len(tms)
        tm_rank_of = {b: r for r, (b, _, _) in enumerate(tms)}
        bases = [b for b, _, _ in tms]
        nested_of = {b: int(any(o != b and b in o for o in bases)) for b in bases}
        info = {}
        for b, s, e in tms:
            for pidx in base2idx[b][:PAGES_PER_TITLE]:
                if pidx not in info and len(info) < MAX_TITLE_CANDS:
                    info[pidx] = (b, s, e)
        cset = dict.fromkeys(info)
        for i in top[:CAND_BM25]:
            cset[int(i)] = None
        for i in di[:CAND_DENSE]:
            cset[int(i)] = None
        cl = list(cset)
        cos = {}
        if USE_DENSE:
            cos = {i: float(dv[d_rank[i]]) for i in cl if i in d_rank}
            miss = [i for i in cl if i not in d_rank]
            if miss:
                vals = np.asarray(DENSE_P[np.array(miss)], dtype=np.float32) @ dq
                cos.update({i: float(v) for i, v in zip(miss, vals)})
        cb = bigrams(claim)
        den, L = max(1, len(cb)), max(1, len(claim))
        rows = []
        for i in cl:
            inf = info.get(i)
            if inf:
                b, s, e = inf
                tm, tl, tf, tp, tr, ne, dg = 1, e - s, (e - s) / L, s / L, tm_rank_of[b], nested_of[b], int(b.isdigit())
            else:
                tm, tl, tf, tp, tr, ne, dg = 0, 0, 0.0, -1.0, n_tm, 0, 0
            bm = float(sc[i])
            dn = cos.get(i, 0.0)
            rows.append([tm, tl, tf, tp, n_tm, tr, ne, int("_(" in titles[i]), len(base2idx.get(base_titles[i], ())), dg,
                         len(base_titles[i]), bm, bm / bmax if bmax > 0 else 0.0, b_rank.get(i, K_BM25),
                         dn, dn - dmax, d_rank.get(i, K_DENSE), math.log1p(plen[i]), len(cb & bigrams(leads[i])) / den])
        rows_all.append(np.asarray(rows, dtype=np.float32).reshape(-1, len(FEATS)))
        pidx_all.append(np.asarray(cl, dtype=np.int64))
        counts.append(len(cl))
    counts = np.array(counts)
    return dict(X=np.concatenate(rows_all), pidx=np.concatenate(pidx_all), counts=counts,
                starts=np.concatenate([[0], np.cumsum(counts)[:-1]]))


def retrieve_features(split):
    path = CACHE_DIR / f"feat_{_sig}_{split['name']}_{len(split['claim'])}.pkl"

    def build():
        if USE_DENSE:
            ds = dense_state()
            Q = encode_dense(split["claim"], ds["tok"], ds["mdl"], desc=f"編碼 {split['name']} claim")
            d_idx, d_val = dense_topk(Q, ds["P"], K_DENSE)
        else:
            Q = d_idx = d_val = None
        return gen_features(split["claim"], Q, d_idx, d_val, desc=f"候選頁面 {split['name']}")
    return cached(path, build)


feats = {s["name"]: retrieve_features(s) for s in (dev, test, train)}
_DENSE.pop("mdl", None)         # 稠密模型之後不再需要，釋放顯存給 BERT 訓練
free_gpu()
for k, v in feats.items():
    print(k, "候選列數：", len(v["X"]), "每題平均候選頁數：", round(len(v["X"]) / len(v["counts"]), 1))

dev 候選列數： 160680 每題平均候選頁數： 53.6
test 候選列數： 334376 每題平均候選頁數： 55.7
train 候選列數： 1279826 每題平均候選頁數： 53.3


In [8]:
# ----- 學習式排序（HistGradientBoosting）與 Recall@K -----
def page_labels(split, F_):
    y = np.zeros(len(F_["X"]), dtype=np.int8)
    for qi, groups in enumerate(split["gold"]):
        gold = {title2idx[t] for g in groups for t, _ in g if t in title2idx}
        s, c = F_["starts"][qi], F_["counts"][qi]
        y[s:s + c] = [int(p in gold) for p in F_["pidx"][s:s + c]]
    return y


def fit_ranker(X, y):
    return HistGradientBoostingClassifier(max_iter=200, learning_rate=0.08, max_leaf_nodes=48, l2_regularization=1.0,
                                          random_state=SEED).fit(X, y)


def to_ranked(F_, scores):
    out = []
    for qi in range(len(F_["counts"])):
        s, c = F_["starts"][qi], F_["counts"][qi]
        o = np.argsort(-scores[s:s + c], kind="stable")
        out.append((F_["pidx"][s:s + c][o], scores[s:s + c][o]))
    return out


Ftr = feats["train"]
y_tr = page_labels(train, Ftr)
qi_rows = np.repeat(np.arange(len(Ftr["counts"])), Ftr["counts"])
sr_rows = np.where(train["label"][qi_rows] != 2)[0]
nei_rows = np.where(train["label"][qi_rows] == 2)[0]
ranker = fit_ranker(Ftr["X"][sr_rows], y_tr[sr_rows])
oof = np.zeros(len(Ftr["X"]), dtype=np.float32)
if len(nei_rows):
    oof[nei_rows] = ranker.predict_proba(Ftr["X"][nei_rows])[:, 1]
for tr_i, va_i in GroupKFold(n_splits=3).split(sr_rows, groups=qi_rows[sr_rows]):   # 訓練題用交叉預測，避免排序分數過度樂觀
    m = fit_ranker(Ftr["X"][sr_rows[tr_i]], y_tr[sr_rows[tr_i]])
    oof[sr_rows[va_i]] = m.predict_proba(Ftr["X"][sr_rows[va_i]])[:, 1]
ranked = {"train": to_ranked(Ftr, oof)}
for name in ("dev", "test"):
    ranked[name] = to_ranked(feats[name], ranker.predict_proba(feats[name]["X"])[:, 1])


def page_recall(split, rk, K):
    hit = n = 0
    for qi, groups in enumerate(split["gold"]):
        if not groups:
            continue
        n += 1
        top = {titles[i] for i in rk[qi][0][:K]}
        hit += any(all(t in top for t, _ in g) for g in groups)
    return hit / max(n, 1)


# 消融：各訊號單獨排序 vs. 學習式排序（候選集相同，所以 K<=20 時 BM25 / 稠密單獨排序等同於原本的 top-K）
Fd, X = feats["dev"], feats["dev"]["X"]
rrf = 1 / (60 + X[:, FI["bm25_rank"]]) + 1 / (60 + X[:, FI["dense_rank"]]) + X[:, FI["tm"]] / (60 + X[:, FI["tm_rank"]])
variants = {
    "只用標題比對": X[:, FI["tm"]] * (1 + X[:, FI["tlen"]]) + 1e-3 * X[:, FI["bm25_rel"]],
    "只用 BM25": X[:, FI["bm25"]],
    "只用稠密檢索": X[:, FI["dense"]] if USE_DENSE else np.zeros(len(X)),
    "RRF 融合（無學習）": rrf,
    "學習式排序（本方法）": None,
}
Ks = (1, 3, 5, 6, 10, 20)
page_table = {}
for name, sc in variants.items():
    rk = ranked["dev"] if sc is None else to_ranked(Fd, sc)
    page_table[name] = [page_recall(dev, rk, K) for K in Ks]
page_df = pd.DataFrame(page_table, index=[f"Recall@{K}" for K in Ks]).T
print("Dev 頁面檢索（完整證據組的所有頁面都在前 K 名才算命中，僅統計 supports/refutes）")
display(page_df.round(4)) if "display" in globals() else print(page_df.round(4))
cand_recall = page_recall(dev, to_ranked(Fd, np.ones(len(X))), 10 ** 9)
print("候選集本身的上限（Recall@全部候選）：", round(cand_recall, 4))

Dev 頁面檢索（完整證據組的所有頁面都在前 K 名才算命中，僅統計 supports/refutes）
             Recall@1  Recall@3  Recall@5  Recall@6  Recall@10  Recall@20
只用標題比對         0.5235    0.7405    0.8255    0.8385     0.8635     0.9000
只用 BM25        0.5400    0.6385    0.6725    0.6845     0.7175     0.7585
只用稠密檢索         0.5235    0.6575    0.6965    0.7135     0.7555     0.7975
RRF 融合（無學習）    0.6995    0.8290    0.8535    0.8570     0.8760     0.9220
學習式排序（本方法）     0.7915    0.8955    0.9175    0.9245     0.9375     0.9410
候選集本身的上限（Recall@全部候選）： 0.9425


## 5. 第二階段：Evidence Retrieval

對每個 claim，取排序後前 `PAGE_TOPK` 頁的句子（先以字元 bigram 重疊度粗篩到 32 句），再交給**微調過的 cross-encoder**（MacBERT）打分。
輸入是 `claim [SEP] 頁面標題：句子`，把標題放進去是因為很多句子以「他」「該」開頭，沒有主詞。

**第二跳（實驗過，最終沒有採用）**：CFEVER 約 10% 的題目需要兩個以上的頁面，且第二個頁面常常沒有出現在 claim 中，而是被第一個證據句「連結」過去
（`lines` 欄位在句子文字之後附有（錨文字, 目標頁面）成對資料）。因此取第一輪分數最高的前 2 句，把它們連結的頁面（最多 6 頁）的候選句也一起打分，
並依 dev 決定第二跳候選的扣分；只有 Recall@5 比不用第二跳多 0.2 個百分點以上才採用。結果最好只多 0.2 個百分點（0.8615 → 0.8635），所以不使用。

訓練資料：正例是標準證據句；負例是同頁面其他句子、被檢索到的頁面的其他句子、以及標準證據句所連結頁面的句子（最難的負例，教模型區分第二跳）。

In [9]:
def build_pool(claim, page_idx, page_prob, pool_size):
    """回傳依粗篩分數排序的候選句 [(pre, pidx, sid, text, links), ...]。"""
    cb = bigrams(claim)
    den = max(1, len(cb))
    cands = []
    for pidx, pp in zip(page_idx, page_prob):
        for sid, text, links in get_sentences(int(pidx)):
            if sid >= PER_PAGE_MAX_SID:
                continue
            cands.append((len(cb & bigrams(text)) / den + 0.5 * float(pp) + (0.1 if sid == 0 else 0.0), int(pidx), sid, text, links))
    cands.sort(key=lambda c: -c[0])
    return cands[:pool_size]


def extra_pool(claim, pages, per_page, exclude=()):
    """指定頁面中，粗篩分數最高的句子（用於第二跳與訓練負例）。"""
    cb = bigrams(claim)
    den = max(1, len(cb))
    cands = []
    for pidx in pages:
        sents = [(len(cb & bigrams(t)) / den + (0.1 if sid == 0 else 0.0), int(pidx), sid, t, lk)
                 for sid, t, lk in get_sentences(int(pidx)) if sid < PER_PAGE_MAX_SID and (titles[int(pidx)], sid) not in exclude]
        sents.sort(key=lambda c: -c[0])
        cands.extend(sents[:per_page])
    cands.sort(key=lambda c: -c[0])
    return cands


# ----- 通用：微調序列分類模型 / 推論 -----
def _new_scaler():
    try:
        return torch.amp.GradScaler("cuda", enabled=USE_AMP)
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=USE_AMP)


def train_seq_clf(examples, num_labels, model_name, max_len, epochs, lr, micro_bs, accum, desc, seed=None, freeze=0, optim="adamw", head_perm=None):
    """examples: [(text_a, text_b, label), ...]。標準流程：AdamW + 線性 warmup/decay + fp16 + 梯度累積。"""
    seed = SEED if seed is None else seed
    torch.manual_seed(seed)                                   # 不同種子 → 分類頭初始化與 dropout 不同（集成需要多樣性）
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels).to(device)
    if head_perm is not None and hasattr(model, "classifier") and model.classifier.out_features == len(head_perm):
        with torch.no_grad():                                 # 把 NLI 分類頭的類別順序換成我們的順序
            pm = torch.tensor(head_perm, device=model.classifier.weight.device)
            model.classifier.weight.copy_(model.classifier.weight[pm].clone())
            model.classifier.bias.copy_(model.classifier.bias[pm].clone())
    if freeze:                                                # 凍結 embedding 與前 freeze 層：省顯存、也省反向傳播時間
        base = model.base_model
        for p in base.embeddings.parameters():
            p.requires_grad = False
        for layer in base.encoder.layer[:freeze]:
            for p in layer.parameters():
                p.requires_grad = False
    if GRAD_CKPT:
        try:
            model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        except TypeError:
            model.gradient_checkpointing_enable()
    params = [p for p in model.parameters() if p.requires_grad]
    if optim == "adafactor":                                  # 優化器狀態極小（無動量、二階矩分解），large 模型才放得進 4GB
        from transformers.optimization import Adafactor
        opt = Adafactor(params, lr=lr, scale_parameter=False, relative_step=False, warmup_init=False, beta1=None, weight_decay=0.0)
    else:
        opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)
    steps = math.ceil(len(examples) / (micro_bs * accum)) * epochs
    sched = get_linear_schedule_with_warmup(opt, int(0.06 * steps), steps)
    scaler = _new_scaler()
    rg = random.Random(seed)
    for ep in range(epochs):
        model.train()
        order = list(range(len(examples)))
        rg.shuffle(order)
        pbar = tqdm(range(0, len(order), micro_bs), desc=f"{desc} epoch {ep + 1}/{epochs}", mininterval=10)
        run_loss, pending = 0.0, 0
        for bi, s in enumerate(pbar):
            batch = [examples[i] for i in order[s:s + micro_bs]]
            enc = tok([b[0] for b in batch], [b[1] for b in batch], truncation=True, max_length=max_len,
                      padding=True, return_tensors="pt")
            enc = {k: v.to(device) for k, v in enc.items()}
            labels = torch.tensor([b[2] for b in batch], device=device)
            with amp_ctx():
                logits = model(**enc).logits
            loss = F.cross_entropy(logits.float(), labels) / accum
            scaler.scale(loss).backward()
            run_loss = 0.98 * run_loss + 0.02 * loss.item() * accum if bi else loss.item() * accum
            pending += 1
            if pending == accum or s + micro_bs >= len(order):
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                scaler.step(opt)
                scaler.update()
                sched.step()
                opt.zero_grad(set_to_none=True)
                pending = 0
            if bi % 20 == 0:
                pbar.set_postfix(loss=round(run_loss, 4))
    model.eval()
    return model, tok


@torch.no_grad()
def score_pairs(bundle, pairs, max_len, bs, desc="推論"):
    """pairs: [(text_a, text_b), ...] -> logits [n, num_labels]（依長度排序分批以加速）。"""
    model, tok = bundle
    model.eval()
    order = np.argsort([len(a) + len(b) for a, b in pairs], kind="stable")
    out = None
    for s in tqdm(range(0, len(order), bs), desc=desc, mininterval=10):
        ids = order[s:s + bs]
        enc = tok([pairs[i][0] for i in ids], [pairs[i][1] for i in ids], truncation=True, max_length=max_len,
                  padding=True, return_tensors="pt")
        enc = {k: v.to(device) for k, v in enc.items()}
        with amp_ctx():
            lg = model(**enc).logits.float().cpu().numpy()
        if out is None:
            out = np.zeros((len(pairs), lg.shape[1]), dtype=np.float32)
        out[ids] = lg
    return out if out is not None else np.zeros((0, 2), dtype=np.float32)


def offload(bundle):
    """把不再使用的模型移到 CPU，釋放 4GB 顯存給下一階段的訓練。"""
    bundle[0].to("cpu")
    free_gpu()


def save_bundle(bundle, path):
    bundle[0].save_pretrained(path)
    bundle[1].save_pretrained(path)


def load_bundle(path):
    return (AutoModelForSequenceClassification.from_pretrained(path).to(device).eval(), AutoTokenizer.from_pretrained(path))


# ----- 小型 LLM 分類器（Qwen3-0.6B-Base + LoRA）-----
# 解碼器模型用「最後一個 token」做分類，所以把輸入寫成：證據 → 論述 → 提示語；超長時只截證據，確保 claim 與提示語完整。
def llm_parts(claim, ctx):
    pre = "證據：\n" + ctx.replace(" [SEP] ", "\n")
    post = "\n\n論述：" + claim + "\n論述與證據的關係（支持／反駁／無法判斷）："
    return pre, post


def llm_ids(tok, pre, post, max_len):
    a = tok(pre, add_special_tokens=False)["input_ids"]
    b = tok(post, add_special_tokens=False)["input_ids"]
    if len(a) + len(b) > max_len:
        a = a[:max(16, max_len - len(b))]
    return (a + b)[-max_len:]


def _llm_pad(batch_ids, pad_id):
    L = max(len(x) for x in batch_ids)
    ids = torch.full((len(batch_ids), L), pad_id, dtype=torch.long)
    att = torch.zeros((len(batch_ids), L), dtype=torch.long)
    for i, x in enumerate(batch_ids):
        ids[i, :len(x)] = torch.tensor(x)
        att[i, :len(x)] = 1
    return ids.to(device), att.to(device)


def _llm_base(name, pad_id):
    try:
        base = AutoModelForSequenceClassification.from_pretrained(name, num_labels=3, dtype=torch.bfloat16)
    except (TypeError, ValueError):
        base = AutoModelForSequenceClassification.from_pretrained(name, num_labels=3, torch_dtype=torch.bfloat16)
    base.config.pad_token_id = pad_id
    return base


def _pad_id(tok):
    return tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id


def train_llm_clf(examples, model_name, max_len, epochs, lr, micro_bs, accum, desc, seed=None, lora_r=16):
    """examples: [(claim, ctx_text, label)]。基底權重 bf16 凍結 + LoRA（fp32），bf16 autocast，不需要 GradScaler。"""
    try:
        from peft import LoraConfig, get_peft_model
    except ImportError as e:
        raise ImportError("需要 peft：在 WSL 的 venv 執行  pip install peft  後重新啟動 kernel") from e
    seed = SEED if seed is None else seed
    torch.manual_seed(seed)
    tok = AutoTokenizer.from_pretrained(model_name)
    pad_id = _pad_id(tok)
    base = _llm_base(model_name, pad_id)
    if GRAD_CKPT:
        try:
            base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        except TypeError:
            base.gradient_checkpointing_enable()
        base.enable_input_require_grads()
    cfg = LoraConfig(task_type="SEQ_CLS", r=lora_r, lora_alpha=2 * lora_r, lora_dropout=0.05,
                     target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
    model = get_peft_model(base, cfg).to(device)
    for p in model.parameters():
        if p.requires_grad:
            p.data = p.data.float()                           # LoRA 與分類頭用 fp32 主權重，避免 bf16 吃掉小的更新量
    params = [p for p in model.parameters() if p.requires_grad]
    print(f"  可訓練參數 {sum(p.numel() for p in params) / 1e6:.1f}M / 總參數 {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M")
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.0)
    steps = math.ceil(len(examples) / (micro_bs * accum)) * epochs
    sched = get_linear_schedule_with_warmup(opt, int(0.05 * steps), steps)
    rg = random.Random(seed)
    for ep in range(epochs):
        model.train()
        order = list(range(len(examples)))
        rg.shuffle(order)
        pbar = tqdm(range(0, len(order), micro_bs), desc=f"{desc} epoch {ep + 1}/{epochs}", mininterval=10)
        run_loss, pending = 0.0, 0
        for bi, s_ in enumerate(pbar):
            batch = [examples[i] for i in order[s_:s_ + micro_bs]]
            ids, att = _llm_pad([llm_ids(tok, *llm_parts(b[0], b[1]), max_len) for b in batch], pad_id)
            labels = torch.tensor([b[2] for b in batch], device=device)
            with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=device.type == "cuda"):
                logits = model(input_ids=ids, attention_mask=att).logits
            loss = F.cross_entropy(logits.float(), labels) / accum
            loss.backward()
            run_loss = 0.98 * run_loss + 0.02 * loss.item() * accum if bi else loss.item() * accum
            pending += 1
            if pending == accum or s_ + micro_bs >= len(order):
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                opt.step()
                sched.step()
                opt.zero_grad(set_to_none=True)
                pending = 0
            if bi % 20 == 0:
                pbar.set_postfix(loss=round(run_loss, 4))
    model.eval()
    return model, tok


def save_llm(bundle, path):
    bundle[0].save_pretrained(path)                           # 只存 LoRA 轉接器與分類頭（很小）
    bundle[1].save_pretrained(path)


def load_llm(path, base_name):
    from peft import PeftModel
    tok = AutoTokenizer.from_pretrained(path)
    model = PeftModel.from_pretrained(_llm_base(base_name, _pad_id(tok)), path).merge_and_unload()
    return model.to(device).eval(), tok


@torch.no_grad()
def score_llm(bundle, parts, max_len, bs, desc="推論"):
    model, tok = bundle
    pad_id = _pad_id(tok)
    seqs = [llm_ids(tok, pre, post, max_len) for pre, post in parts]
    order = np.argsort([len(x) for x in seqs], kind="stable")
    out = np.zeros((len(seqs), 3), dtype=np.float32)
    for s_ in tqdm(range(0, len(order), bs), desc=desc, mininterval=10):
        idx = order[s_:s_ + bs]
        ids, att = _llm_pad([seqs[i] for i in idx], pad_id)
        with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=device.type == "cuda"):
            out[idx] = model(input_ids=ids, attention_mask=att).logits.float().cpu().numpy()
    return out

In [10]:
# ----- 訓練 cross-encoder 重排器 -----
def build_rerank_examples(split, rk, claim_ids):
    rg = random.Random(SEED)
    ex = []
    for qi in tqdm(claim_ids, desc="建立重排訓練資料", mininterval=5):
        groups = split["gold"][qi]
        claim = split["claim"][qi]
        gold = {(t, s) for g in groups for t, s in g}
        pos, gold_pages = [], []
        for t, s in gold:
            j = title2idx.get(t)
            if j is None:
                continue
            gold_pages.append(j)
            for sid, text, links in get_sentences(j):
                if sid == s:
                    pos.append((claim, passage(j, text), 1))
        if not pos:
            continue
        idxs, probs = rk[qi]
        pool = build_pool(claim, idxs[:PAGE_TOPK], probs[:PAGE_TOPK], 40)
        negs = {(c[1], c[2]): c for c in pool if (titles[c[1]], c[2]) not in gold}
        for c in extra_pool(claim, set(gold_pages), 6, exclude=gold):                     # 同一個標準頁面的其他句子
            negs.setdefault((c[1], c[2]), c)
        hop_pages = set()
        for t, s in gold:
            j = title2idx.get(t)
            for sid, _, links in (get_sentences(j) if j is not None else ()):
                if sid == s:
                    hop_pages.update(title2idx[l] for l in links if l in title2idx)
        for c in extra_pool(claim, hop_pages - set(gold_pages), 3, exclude=gold)[:6]:      # 標準證據句連結到的頁面（第二跳負例）
            negs.setdefault((c[1], c[2]), c)
        neg_list = sorted(negs.values(), key=lambda c: -c[0])
        chosen = neg_list[:3] + rg.sample(neg_list[3:], min(RERANK_NEG - 3, len(neg_list[3:])))
        ex.extend(pos)
        ex.extend((claim, passage(c[1], c[3]), 0) for c in chosen)
    rg.shuffle(ex)
    return ex


_rr_path = CACHE_DIR / "reranker"
if _rr_path.exists() and not FORCE_RETRAIN:
    reranker = load_bundle(_rr_path)
    print("載入已訓練的重排器：", _rr_path)
else:
    sr_ids = [i for i in range(len(train["claim"])) if train["label"][i] != 2]
    random.Random(SEED).shuffle(sr_ids)
    sr_ids = sr_ids[:RERANK_TRAIN_CLAIMS] if RERANK_TRAIN_CLAIMS else sr_ids
    rr_examples = build_rerank_examples(train, ranked["train"], sr_ids)
    print("重排訓練資料：", len(rr_examples), "筆（正例", sum(e[2] for e in rr_examples), "）")
    reranker = train_seq_clf(rr_examples, 2, RERANK_MODEL, RERANK_MAX_LEN, RERANK_EPOCHS, RERANK_LR, RERANK_BS, RERANK_ACCUM, "訓練重排器")
    save_bundle(reranker, _rr_path)
    del rr_examples
free_gpu()
RR_TAG = str(int((_rr_path / "config.json").stat().st_mtime))   # 重排器重訓後，舊的重排快取自動失效

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:  10%|▉         | 20/201 [00:00<00:00, 192.86it/s]

Loading weights:  25%|██▍       | 50/201 [00:00<00:00, 253.61it/s]

Loading weights:  38%|███▊      | 76/201 [00:00<00:00, 249.64it/s]

Loading weights:  51%|█████     | 102/201 [00:00<00:00, 240.26it/s]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 403.86it/s]

載入已訓練的重排器： output/cache/reranker


In [11]:
# ----- 重排推論（含第二跳）-----
def rerank_split(split, rk, desc):
    """回傳 (first, final)：first 只含第一跳；final 加上第二跳。每題是 [(score, pidx, sid, text, links), ...] 由高到低。"""
    n = len(split["claim"])
    pools = [build_pool(split["claim"][qi], rk[qi][0][:PAGE_TOPK], rk[qi][1][:PAGE_TOPK], RERANK_POOL) for qi in range(n)]
    pairs = [(split["claim"][qi], passage(c[1], c[3])) for qi in range(n) for c in pools[qi]]
    own = [(qi, k) for qi in range(n) for k in range(len(pools[qi]))]
    lg = score_pairs(reranker, pairs, RERANK_MAX_LEN, INFER_BS, desc)
    first = [[] for _ in range(n)]
    sc = (lg[:, 1] - lg[:, 0]) if len(lg) else np.zeros(0)
    for (qi, k), v in zip(own, sc):
        c = pools[qi][k]
        first[qi].append((float(v), c[1], c[2], c[3], c[4]))
    first = [sorted(x, key=lambda t: -t[0]) for x in first]
    if not USE_HOP2:
        return first, first
    hop_pools, hop_pairs, hop_own = [], [], []
    for qi in range(n):
        have = {c[1] for c in pools[qi]}
        targets = []
        for _, pidx, sid, text, links in first[qi][:HOP_SRC]:
            for t in links:
                j = title2idx.get(t)
                if j is not None and j not in have and j not in targets:
                    targets.append(j)
        hp = extra_pool(split["claim"][qi], targets[:HOP_MAX_PAGES], HOP_PER_PAGE)[:HOP_POOL]
        hop_pools.append(hp)
        for k, c in enumerate(hp):
            hop_pairs.append((split["claim"][qi], passage(c[1], c[3])))
            hop_own.append((qi, k))
    final = [list(x) for x in first]
    if hop_pairs:
        hl = score_pairs(reranker, hop_pairs, RERANK_MAX_LEN, INFER_BS, desc + "（第二跳）")
        for (qi, k), v in zip(hop_own, hl[:, 1] - hl[:, 0]):
            c = hop_pools[qi][k]
            final[qi].append((float(v), c[1], c[2], c[3], c[4]))
    final = [sorted(x, key=lambda t: -t[0]) for x in final]
    return first, final


def evidence_recall(split, lists, K=5):
    hit = n = 0
    for qi, groups in enumerate(split["gold"]):
        if not groups:
            continue
        n += 1
        top = {(titles[c[1]], c[2]) for c in lists[qi][:K]}
        hit += any(all(x in top for x in g) for g in groups)
    return hit / max(n, 1)


rr = {}
for s in (dev, test):
    rr[s["name"]] = cached(CACHE_DIR / f"rerank_{_sig}_{RR_TAG}_{s['name']}_{len(s['claim'])}.pkl",
                           lambda s=s: rerank_split(s, ranked[s["name"]], f"重排 {s['name']}"))


def merge_hop(first, final, delta):
    """第二跳候選（final 中不在 first 的句子）扣掉 delta 分後與第一跳合併；delta=None 代表不使用第二跳。"""
    if delta is None:
        return first
    seen = {(c[1], c[2]) for c in first}
    out = list(first) + [(c[0] - delta,) + tuple(c[1:]) for c in final if (c[1], c[2]) not in seen]
    return sorted(out, key=lambda t: -t[0])


ev_rows = {}
for d in HOP_PENALTIES:
    lists = [merge_hop(a, b, d) for a, b in zip(*rr["dev"])]
    ev_rows["不使用第二跳" if d is None else f"第二跳（扣 {d:g} 分）"] = [evidence_recall(dev, lists, K) for K in (1, 3, 5)]
ev_df = pd.DataFrame(ev_rows, index=["Recall@1", "Recall@3", "Recall@5"]).T
print("Dev 證據句檢索（前 K 句需完整涵蓋任一組標準證據；僅統計 supports/refutes）")
display(ev_df.round(4)) if "display" in globals() else print(ev_df.round(4))

# 第二跳的句子容易擠掉第一跳的正確句子，所以只有在 Recall@5 明顯提升（>= 0.2 個百分點）時才採用
HOP_DELTA, _best = None, evidence_recall(dev, rr["dev"][0], 5)
for d in HOP_PENALTIES:
    if d is not None and ev_rows[f"第二跳（扣 {d:g} 分）"][2] > _best + 0.002:
        HOP_DELTA, _best = d, ev_rows[f"第二跳（扣 {d:g} 分）"][2]
print("採用的第二跳扣分：", HOP_DELTA, "（None = 不使用第二跳）")
EV = {n: [merge_hop(a, b, HOP_DELTA) for a, b in zip(*rr[n])] for n in ("dev", "test")}   # 之後都使用這份證據排序
for ng in (1, 2, 3):   # 依標準證據組的句子數分析
    ids = [i for i, gs in enumerate(dev["gold"]) if gs and min(len(g) for g in gs) == ng]
    if ids:
        sub = dict(gold=[dev["gold"][i] for i in ids])
        print(f"  最短證據組含 {ng} 句：{len(ids)} 題，Recall@5 =", round(evidence_recall(sub, [EV["dev"][i] for i in ids], 5), 4))

Dev 證據句檢索（前 K 句需完整涵蓋任一組標準證據；僅統計 supports/refutes）
            Recall@1  Recall@3  Recall@5
第二跳（扣 0 分）    0.4885    0.7865    0.8540
第二跳（扣 1 分）    0.4945    0.7965    0.8630
第二跳（扣 2 分）    0.4955    0.8015    0.8635
第二跳（扣 3 分）    0.4955    0.8020    0.8635
第二跳（扣 5 分）    0.4955    0.8015    0.8635
不使用第二跳        0.4955    0.8005    0.8615
採用的第二跳扣分： None （None = 不使用第二跳）
  最短證據組含 1 句：1273 題，Recall@5 = 0.9356
  最短證據組含 2 句：513 題，Recall@5 = 0.7856
  最短證據組含 3 句：169 題，Recall@5 = 0.6509


## 5b. 第二個重排器（v6.0）

句子重排器（MacBERT）只訓練 1 epoch，而且只有它一個模型在決定證據。這一節訓練**第二個重排器**，與第一個互補：
- 骨幹不同（`chinese-roberta-wwm-ext`），訓練 2 epoch、輸入長度 160；
- **hard negative mining**：負例直接取自第一個重排器對訓練題實際排出的高分非標準句（前 3 名）+ 同一標準頁面的其他句子 + 少量隨機負例，專門教它分辨第一個重排器容易搞混的句子。

第二個重排器**只拿來給下一節的選擇器當特徵**：分類器的輸入、所有分類快取都不變，所以不需要重訓任何分類器。

In [12]:
USE_RR2 = True
RR2_MODEL = "hfl/chinese-roberta-wwm-ext"
RR2_MAX_LEN, RR2_EPOCHS, RR2_LR, RR2_BS, RR2_ACCUM = 160, 2, 3e-5, 16, 2
RR2_HARD, RR2_RAND, RR2_SIB = 3, 2, 2                 # 每題負例：第一個重排器的前 3 名非標準句 + 2 個隨機 + 同頁 2 句
SEL_N = 12                                            # 選擇器只重新排序重排器的前 12 名；第二個重排器也只替這 12 句打分


def build_rerank2_examples(ids, lists):
    rg = random.Random(SEED + 7)
    ex = []
    for pos_, qi in enumerate(tqdm(ids, desc="建立重排器2訓練資料", mininterval=5)):
        claim = train["claim"][qi]
        gold = {(t, s) for g in train["gold"][qi] for t, s in g}
        pos_txt, gold_pages = [], []
        for t, s in gold:
            j = title2idx.get(t)
            if j is None:
                continue
            gold_pages.append(j)
            pos_txt += [passage(j, text) for sid, text, _ in get_sentences(j) if sid == s]
        if not pos_txt:
            continue
        negs = [c for c in lists[pos_] if (titles[c[1]], c[2]) not in gold]
        chosen = {(c[1], c[2]): c for c in negs[:RR2_HARD]}                                   # hard negatives：第一個重排器最愛的錯誤答案
        rest = negs[RR2_HARD:RR2_HARD + 12]
        for c in rg.sample(rest, min(RR2_RAND, len(rest))):
            chosen.setdefault((c[1], c[2]), c)
        for c in extra_pool(claim, set(gold_pages), RR2_SIB, exclude=gold)[:RR2_SIB]:         # 標準證據同頁的其他句子
            chosen.setdefault((c[1], c[2]), c)
        ex.extend((claim, p, 1) for p in pos_txt)
        ex.extend((claim, passage(c[1], c[3]), 0) for c in chosen.values())
    rg.shuffle(ex)
    return ex


if USE_RR2:
    _rr2_path = CACHE_DIR / "reranker2"
    if _rr2_path.exists() and not FORCE_RETRAIN:
        reranker2 = load_bundle(_rr2_path)
        print("載入已訓練的重排器2：", _rr2_path)
    else:
        _ids2 = [i for i in range(len(train["claim"])) if train["label"][i] != 2]            # 與分類訓練資料相同的題目集合 → 共用第一個重排器的重排快取
        if CLS_TRAIN_SR_CLAIMS:
            random.Random(SEED).shuffle(_ids2)
            _ids2 = sorted(_ids2[:CLS_TRAIN_SR_CLAIMS])
        _first, _final = cached(CACHE_DIR / f"rerank_{_sig}_{RR_TAG}_trainSR_{len(_ids2)}.pkl",
                                lambda: rerank_split(dict(claim=[train["claim"][i] for i in _ids2]), [ranked["train"][i] for i in _ids2], "重排 train-SR"))
        _lists2 = [merge_hop(a, b, HOP_DELTA) for a, b in zip(_first, _final)]
        rr2_examples = build_rerank2_examples(_ids2, _lists2)
        print("重排器2 訓練資料：", len(rr2_examples), "筆（正例", sum(e[2] for e in rr2_examples), "）")
        offload(reranker)                                                                     # 騰出顯存給第二個重排器
        reranker2 = train_seq_clf(rr2_examples, 2, RR2_MODEL, RR2_MAX_LEN, RR2_EPOCHS, RR2_LR, RR2_BS, RR2_ACCUM, "訓練重排器2", seed=SEED + 7)
        save_bundle(reranker2, _rr2_path)
        del rr2_examples, _first, _final, _lists2
        reranker[0].to(device)                                                                # 第一個重排器還要用（若分類訓練資料沒有快取）
        free_gpu()
    RR2_TAG = str(int((_rr2_path / "config.json").stat().st_mtime))


def rr2_scores(split, lists):
    """第二個重排器替每題 lists[qi][:SEL_N] 打分（logit 差），回傳每題一個 numpy 陣列；有快取。"""
    def build():
        pairs, own = [], []
        for qi, lst in enumerate(lists):
            for k, c in enumerate(lst[:SEL_N]):
                pairs.append((split["claim"][qi], passage(c[1], c[3])))
                own.append((qi, k))
        lg = score_pairs(reranker2, pairs, RR2_MAX_LEN, INFER_BS, f"重排器2 {split['name']}")
        out = [np.zeros(min(SEL_N, len(l)), dtype=np.float32) for l in lists]
        for (qi, k), v in zip(own, lg[:, 1] - lg[:, 0]):
            out[qi][k] = v
        return out
    return cached(CACHE_DIR / f"rr2score_{_sig}_{RR_TAG}_{RR2_TAG}_{HOP_DELTA}_{SEL_N}_{split['name']}_{len(lists)}.pkl", build)


if USE_RR2:
    RR2_S = {s["name"]: rr2_scores(s, EV[s["name"]]) for s in (dev, test)}
    offload(reranker2)
    free_gpu()
    _g = [(dev["gold"][qi], EV["dev"][qi][:SEL_N], RR2_S["dev"][qi]) for qi in range(len(dev["claim"]))]
    _by2 = [[c for _, c in sorted(zip(-s, lst), key=lambda t: t[0])] for _, lst, s in _g]      # 只用第二個重排器排序（診斷）
    print("Dev 證據 Recall@1/3/5（第一個重排器）：", [round(evidence_recall(dev, EV["dev"], K), 4) for K in (1, 3, 5)])
    print("Dev 證據 Recall@1/3/5（只用第二個重排器重排前 12 句）：", [round(evidence_recall(dev, _by2, K), 4) for K in (1, 3, 5)])

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:  11%|█         | 22/201 [00:00<00:00, 214.18it/s]

Loading weights:  22%|██▏       | 44/201 [00:00<00:00, 200.77it/s]

Loading weights:  36%|███▋      | 73/201 [00:00<00:00, 238.51it/s]

Loading weights:  49%|████▉     | 99/201 [00:00<00:00, 241.21it/s]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 401.16it/s]

載入已訓練的重排器2： output/cache/reranker2


Dev 證據 Recall@1/3/5（第一個重排器）： [0.4955, 0.8005, 0.8615]
Dev 證據 Recall@1/3/5（只用第二個重排器重排前 12 句）： [0.512, 0.7875, 0.8585]


## 5c. 證據句重新挑選（Evidence re-selection）

觀察：分類準確率 0.77，但 CFEVER 只有 0.73，差距來自「前 5 句沒有涵蓋完整的標準證據組」。候選池其實涵蓋 ~96% 的題目，
而 Recall@5 只有 ~86%；其中約 2/3 的失敗來自需要多句才能判斷的題目。重排器只看「claim + 單一句子」，不知道同頁其他句子、頁面排名、句子位置。

這一步用 dev 的 S/R 題訓練一個 HistGradientBoosting 句子選擇器，目標是「該句是否屬於任一組標準證據」。特徵分三組：
- **base**：重排分數、名次、與第 1 名的分差、句子編號、該頁的頁面機率/名次、同頁在前 5 名的句數、與 claim 的字元二元組重疊與新增涵蓋量…；
- **link**：多句證據的關聯——是否被第 1/2 名句子的超連結指到、與第 1 名同頁時的句子距離、相鄰句子是否也在候選中；
- **rr2**：第二個重排器的分數、名次與分差。

dev 用 5 折（以題目為單位）out-of-fold 預測評估三種特徵組合，**取 OOF Recall@5 最高者；比較簡單的組合只有在被超過 0.1 個百分點時才會被取代；
最後只在比原重排器提升 >= 0.5 個百分點時才採用**。採用時只改變「送出的 5 句證據」，分類器的輸入仍沿用第一個重排器的前 4 句。

In [13]:
SEL_MIN_GAIN = 0.005
N_BASE, N_LINK, N_RR2 = 21, 4, 6


def _bg(s):
    s = re.sub(r"\s+", "", s)
    return {s[i:i + 2] for i in range(len(s) - 1)}


def sel_features(split, lists, rk, s2=None):
    """回傳 (X, meta)；X 的欄位依序為 base(21) / link(4) / rr2(6)；meta 為 (題目編號, 在 lists[qi] 中的位置)。"""
    X, meta = [], []
    for qi, lst in enumerate(lists):
        cand = lst[:SEL_N]
        if not cand:
            continue
        claim = split["claim"][qi]
        cb = _bg(claim)
        rank_of = {int(p): r for r, p in enumerate(rk[qi][0][:200])}
        prob_of = {int(p): float(v) for p, v in zip(rk[qi][0][:200], rk[qi][1][:200])}
        pages = []
        for c in cand:
            if c[1] not in pages:
                pages.append(c[1])
        s0 = cand[0][0]
        s1 = cand[1][0] if len(cand) > 1 else s0 - 1.0
        top_page = cand[0][1]
        links1 = set(cand[0][4])
        links2 = set(cand[1][4]) if len(cand) > 1 else set()
        keys = {(c[1], c[2]) for c in cand}
        if s2 is not None:
            v2 = np.asarray(s2[qi], dtype=np.float64)
            r2 = np.argsort(np.argsort(-v2, kind="stable"), kind="stable")
            m2 = float(v2.max())
            m2b = float(np.sort(v2)[-2]) if len(v2) > 1 else m2 - 1.0
            p2top = cand[int(v2.argmax())][1]
        covered = set()
        for k, c in enumerate(cand):
            sc, p, sid, text = c[0], c[1], c[2], c[3]
            same = [d for d in cand if d[1] == p]
            same_sid = [d[2] for d in same]
            ttl = _PAREN.sub("", titles[p]).replace("_", " ")
            bt = _bg(text)
            inter = cb & bt
            row = [sc, k, sc - s0, sc - s1, sid, float(sid == 0), math.log1p(sid),
                   rank_of.get(int(p), 200), prob_of.get(int(p), 0.0), pages.index(p), max(d[0] for d in same) - s0,
                   sum(1 for d in cand[:5] if d[1] == p), len(same), sid - min(same_sid),
                   float(p == top_page), len(text), len(claim), float(ttl in claim), len(ttl),
                   len(inter) / max(len(cb), 1), len(inter - covered) / max(len(cb), 1)]
            row += [float(titles[p] in links1 and p != top_page), float(titles[p] in links2 and p != cand[1][1]),
                    abs(sid - cand[0][2]) if p == top_page else 99.0,
                    float((p, sid - 1) in keys) + float((p, sid + 1) in keys)]
            if s2 is not None:
                row += [v2[k], int(r2[k]), v2[k] - m2, v2[k] - m2b, float(p == p2top), (k + int(r2[k])) / 2.0]
            else:
                row += [0.0] * N_RR2
            X.append(row)
            covered |= inter
            meta.append((qi, k))
    return np.array(X, dtype=np.float32), meta


def sel_reorder(lists, meta, prob):
    """依選擇器機率重新排序前 SEL_N 名，其餘維持原順序接在後面。"""
    pq = defaultdict(dict)
    for (qi, k), p in zip(meta, prob):
        pq[qi][k] = p
    out = []
    for qi, lst in enumerate(lists):
        if qi not in pq:
            out.append(lst)
            continue
        order = sorted(pq[qi], key=lambda k: (-pq[qi][k], k))
        out.append([lst[k] for k in order] + list(lst[SEL_N:]))
    return out


def sel_model():
    return HistGradientBoostingClassifier(max_iter=250, learning_rate=0.05, max_leaf_nodes=15, min_samples_leaf=30,
                                          l2_regularization=1.0, random_state=SEED)


SEL_SETS = {"base": np.arange(N_BASE), "base+link": np.arange(N_BASE + N_LINK)}
if USE_RR2:
    SEL_SETS["base+link+rr2"] = np.arange(N_BASE + N_LINK + N_RR2)
X_dev, meta_dev = sel_features(dev, EV["dev"], ranked["dev"], RR2_S["dev"] if USE_RR2 else None)
_gold_dev = [{x for g in dev["gold"][qi] for x in g} for qi in range(len(dev["claim"]))]
y_dev = np.array([(titles[EV["dev"][qi][k][1]], EV["dev"][qi][k][2]) in _gold_dev[qi] for qi, k in meta_dev])
q_dev = np.array([qi for qi, _ in meta_dev])
sr_mask = np.array([dev["label"][qi] != 2 for qi in q_dev])
_folds = list(GroupKFold(5).split(X_dev, y_dev, q_dev))
SEL_OOF, SEL_LISTS = {}, {}
for _nm, _cols in SEL_SETS.items():
    _oof = np.zeros(len(y_dev))
    for tr_i, va_i in _folds:
        tr_i = tr_i[sr_mask[tr_i]]                        # 只用 supports/refutes 題訓練（NEI 沒有標準證據）
        _oof[va_i] = sel_model().fit(X_dev[tr_i][:, _cols], y_dev[tr_i]).predict_proba(X_dev[va_i][:, _cols])[:, 1]
    SEL_LISTS[_nm] = sel_reorder(EV["dev"], meta_dev, _oof)
    SEL_OOF[_nm] = {K: round(evidence_recall(dev, SEL_LISTS[_nm], K), 4) for K in (1, 3, 5)}
print("證據句 Recall@1/3/5（dev，選擇器為 5 折 OOF）")
print("  重排器 baseline        ", {K: round(evidence_recall(dev, EV["dev"], K), 4) for K in (1, 3, 5)})
for _nm in SEL_SETS:
    print(f"  選擇器 {_nm:14s}", SEL_OOF[_nm])
_base5 = evidence_recall(dev, EV["dev"], 5)
SEL_SET, _best = None, _base5 + SEL_MIN_GAIN - 1e-9
for _nm in SEL_SETS:                                      # 由簡單到複雜；複雜的組合要多贏 0.1 個百分點才取代
    if SEL_OOF[_nm][5] >= _best + (0.001 if SEL_SET is not None else 0.0):
        SEL_SET, _best = _nm, SEL_OOF[_nm][5]
SEL_ON = SEL_SET is not None
print("採用證據句重新挑選：", SEL_ON, "| 特徵組合：", SEL_SET)
for ng in (1, 2, 3):
    ids = [i for i, gs in enumerate(dev["gold"]) if gs and min(len(g) for g in gs) == ng]
    if ids:
        sub = dict(gold=[dev["gold"][i] for i in ids])
        _sel_l = SEL_LISTS[SEL_SET] if SEL_ON else EV["dev"]
        print(f"  最短證據組 {ng} 句（{len(ids)} 題）Recall@5：",
              round(evidence_recall(sub, [EV["dev"][i] for i in ids], 5), 4), "→", round(evidence_recall(sub, [_sel_l[i] for i in ids], 5), 4))
if SEL_ON:
    _cols = SEL_SETS[SEL_SET]
    _sel_final = sel_model().fit(X_dev[sr_mask][:, _cols], y_dev[sr_mask])
    X_te, meta_te = sel_features(test, EV["test"], ranked["test"], RR2_S["test"] if USE_RR2 else None)
    EV_SEL_TEST = sel_reorder(EV["test"], meta_te, _sel_final.predict_proba(X_te[:, _cols])[:, 1])
EV_OUT = {"dev": SEL_LISTS[SEL_SET] if SEL_ON else EV["dev"], "test": EV_SEL_TEST if SEL_ON else EV["test"]}   # 只用於「送出的證據」與 CFEVER 計算

證據句 Recall@1/3/5（dev，選擇器為 5 折 OOF）
  重排器 baseline         {1: 0.4955, 3: 0.8005, 5: 0.8615}
  選擇器 base           {1: 0.497, 3: 0.8275, 5: 0.878}
  選擇器 base+link      {1: 0.501, 3: 0.8285, 5: 0.878}
  選擇器 base+link+rr2  {1: 0.5105, 3: 0.8305, 5: 0.878}
採用證據句重新挑選： True | 特徵組合： base
  最短證據組 1 句（1273 題）Recall@5： 0.9356 → 0.9442
  最短證據組 2 句（513 題）Recall@5： 0.7856 → 0.8129
  最短證據組 3 句（169 題）Recall@5： 0.6509 → 0.6864


## 6. 第三階段：Classification

輸入 `[CLS] claim [SEP] 標題：證據1 [SEP] 標題：證據2 … [SEP]`（前 4 句，依重排分數由高到低；97% 的標準證據組不超過 3 句），MacBERT 分三類。

訓練資料的證據要貼近測試時的分佈：
**關鍵：每一類的訓練輸入都必須長得跟測試時一樣**，否則模型會學到與標籤無關的捷徑。第一版（v0）犯了這個錯：
NEI 的證據永遠是 4 句，但 supports/refutes 是「標準證據 + 隨機 0~3 句干擾」，大約一半不足 4 句，而且標準證據永遠排第一。
結果模型學成「滿 4 句 = NEI」，測試時每題都是 4 句，於是 dev 上 43% 的 supports/refutes 被判成 NEI。

現在三類全部用**同一個重排器對訓練題實際重排的結果**建立證據：
- **所有題目**：取重排後的前 4 句，依重排分數排序（有 30% 機率打亂順序）。
- **supports / refutes**：若標準證據不在前 4 句內，就把缺的標準證據句放進去、擠掉分數最低的非標準句，確保這題「看得到證據」。
- **NEI**：沒有標準證據，直接用重排器挑出的前 4 句。

最終模型是**多個分類器的集成**（成員清單見第 1 節的 `CLS_MEMBERS` / `ACTIVE_MEMBERS`）：各成員 log-softmax 後等權平均，再對 NOT ENOUGH INFO 的 logit 加偏差。
偏差的選擇見第 7 節：dev（三類各 1/3）與 test 的類別比例不同，dev 上的校準（2 個參數、二折交叉驗證）只當參考，部署的偏差（1.5 / 2.0）是依 public 曲線決定的。

In [14]:
def ctx_text(items):
    return " [SEP] ".join(passage(p, t) for p, _, t in items)


def build_cls_examples(seed=None):
    rg = random.Random(SEED if seed is None else seed)
    nei_ids = [i for i in range(len(train["claim"])) if train["label"][i] == 2]
    sr_ids = [i for i in range(len(train["claim"])) if train["label"][i] != 2]
    if CLS_TRAIN_SR_CLAIMS:
        rg.shuffle(sr_ids)
        sr_ids = sorted(sr_ids[:CLS_TRAIN_SR_CLAIMS])

    def rerank_ids(ids, tag):
        sub = dict(claim=[train["claim"][i] for i in ids])
        first, final = cached(CACHE_DIR / f"rerank_{_sig}_{RR_TAG}_train{tag}_{len(ids)}.pkl",
                              lambda: rerank_split(sub, [ranked["train"][i] for i in ids], f"重排 train-{tag}"))
        return [merge_hop(a, b, HOP_DELTA) for a, b in zip(first, final)]

    nei_lists, sr_lists = rerank_ids(nei_ids, "NEI"), rerank_ids(sr_ids, "SR")
    ex = []
    for pos_, qi in enumerate(tqdm(sr_ids, desc="建立分類訓練資料(S/R)", mininterval=5)):
        lst = sr_lists[pos_]
        groups = train["gold"][qi]
        best = min(groups, key=lambda g: (len({t for t, _ in g}), len(g)))
        by_key = {(titles[c[1]], c[2]): c for c in lst}
        top_score = lst[0][0] if lst else 0.0
        gold_items = []
        for t, s in best:
            if (t, s) in by_key:
                gold_items.append(by_key[(t, s)])
                continue
            j = title2idx.get(t)                       # 標準證據不在候選池：以最高分注入，確保這題看得到證據
            txt = next((x for sid, x, _ in (get_sentences(j) if j is not None else ()) if sid == s), None)
            if txt is None:
                gold_items = None
                break
            gold_items.append((top_score, j, s, txt, ()))
        if not gold_items:
            continue
        all_gold = {(t, s) for g in groups for t, s in g}
        fill = [c for c in lst if (titles[c[1]], c[2]) not in all_gold][:max(0, CLS_CTX_K - len(gold_items))]
        ctx = sorted(gold_items + fill, key=lambda c: -c[0])[:CLS_CTX_K]
        if rg.random() < CLS_SHUFFLE_P:
            rg.shuffle(ctx)
        ex.append((train["claim"][qi], ctx_text([(c[1], c[2], c[3]) for c in ctx]), int(train["label"][qi])))
    for pos_, qi in enumerate(nei_ids):
        ctx = list(nei_lists[pos_][:CLS_CTX_K])
        if ctx:
            if rg.random() < CLS_SHUFFLE_P:
                rg.shuffle(ctx)
            ex.append((train["claim"][qi], ctx_text([(c[1], c[2], c[3]) for c in ctx]), 2))
    rg.shuffle(ex)
    n_ctx = collections.Counter(e[1].count(" [SEP] ") + 1 for e in ex)
    print("每筆訓練資料的證據句數分佈（三類應該一致）：", dict(sorted(n_ctx.items())))
    return ex


_cls_path = CACHE_DIR / "classifier_v2"          # v2：修正訓練/測試證據分佈不一致的問題（v0 的 classifier 不再使用）
if _cls_path.exists() and not (FORCE_RETRAIN or FORCE_RETRAIN_CLASSIFIER):
    classifier = load_bundle(_cls_path)
    cls_prior = np.array(json.loads((_cls_path / "prior.json").read_text()))
    print("載入已訓練的分類器：", _cls_path)
else:
    cls_examples = build_cls_examples()
    offload(reranker)                      # 重排器之後不再需要，釋放顯存
    cls_prior = np.bincount([e[2] for e in cls_examples], minlength=3) / len(cls_examples)
    print("分類訓練資料：", len(cls_examples), "筆，類別比例 supports/refutes/NEI =", cls_prior.round(3))
    classifier = train_seq_clf(cls_examples, 3, CLS_MODEL, CLS_MAX_LEN, CLS_EPOCHS, CLS_LR, CLS_BS, CLS_ACCUM, "訓練分類器")
    save_bundle(classifier, _cls_path)
    (_cls_path / "prior.json").write_text(json.dumps(cls_prior.tolist()))
    del cls_examples
offload(classifier)                        # 之後每個成員都是「用到才從硬碟載入」，一次只佔一個模型的顯存
free_gpu()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:  15%|█▍        | 30/201 [00:00<00:00, 295.77it/s]

Loading weights:  30%|██▉       | 60/201 [00:00<00:00, 285.63it/s]

Loading weights:  44%|████▍     | 89/201 [00:00<00:00, 283.41it/s]

Loading weights:  59%|█████▊    | 118/201 [00:00<00:00, 267.29it/s]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 436.72it/s]

載入已訓練的分類器： output/cache/classifier_v2


In [15]:
# ----- 分類器集成：成員訓練、逐成員推論（logits 有快取）-----
offload(reranker)                          # 重排器之後不再使用（訓練資料的重排結果都已快取）
free_gpu()
# 先看 claim + 前 4 句證據的 token 數：若常超過 256，會把後面的證據截掉（新成員會自動改用 384）
try:
    _tk = AutoTokenizer.from_pretrained(CLS_MODEL)
    _tk.model_max_length = 100000
    _dp = [(dev["claim"][qi], ctx_text([(c[1], c[2], c[3]) for c in EV["dev"][qi][:CLS_CTX_K]])) for qi in range(len(dev["claim"]))]
    _ln = np.array([len(x) for x in _tk([a for a, _ in _dp], [b for _, b in _dp], truncation=False)["input_ids"]])
    TRUNC_RATE = float((_ln > CLS_MAX_LEN).mean())
    print(f"dev 的輸入長度（token）：中位數 {np.median(_ln):.0f}、90 百分位 {np.percentile(_ln, 90):.0f}；超過 {CLS_MAX_LEN} 而被截斷的比例 = {TRUNC_RATE:.3f}")
except Exception as e:
    print("（略過輸入長度檢查：", repr(e)[:80], "）")
    TRUNC_RATE = 0.0
EXT_LEN = 384 if TRUNC_RATE > 0.10 else CLS_MAX_LEN        # 新成員使用的最大長度
OOM_ERR = getattr(torch.cuda, "OutOfMemoryError", RuntimeError)


def member_path(m):
    return CACHE_DIR / (m.get("path") or f"cls_{m['tag']}")


def train_member(m):
    path = member_path(m)
    if path.exists() and not (FORCE_RETRAIN or FORCE_RETRAIN_CLASSIFIER):
        return
    ex = build_cls_examples(seed=m.get("seed"))
    if m.get("kind") == "llm":
        if m.get("n_train"):
            ex = ex[:m["n_train"]]
        print(f"  {m['tag']}：{len(ex)} 筆訓練資料；留意進度條的速度，可估算全部訓練時間")
        bundle = train_llm_clf(ex, m["model"], m.get("max_len", 384), m.get("epochs", 2), m.get("lr", 1e-4), m.get("bs", 4),
                               m.get("accum", 4), f"訓練 {m['tag']}", seed=m.get("seed"))
        save_llm(bundle, path)
        (path / "meta.json").write_text(json.dumps(dict(kind="llm", max_len=m.get("max_len", 384), base=m["model"])))
        offload(bundle); del bundle, ex
        free_gpu()
        return
    if m.get("swap"):
        ex = [(b, a, y) for a, b, y in ex]                    # (證據, claim)：NLI 的 前提/假設 順序
    freeze, max_len = m.get("freeze", 0), m.get("max_len", EXT_LEN)
    while True:
        oom, bundle = False, None
        try:
            bundle = train_seq_clf(ex, 3, m["model"], max_len, m.get("epochs", CLS_EPOCHS), m.get("lr", CLS_LR), m.get("bs", CLS_BS),
                                   m.get("accum", CLS_ACCUM), f"訓練 {m['tag']}", seed=m.get("seed"), freeze=freeze, optim=m.get("optim", "adamw"), head_perm=m.get("head_perm"))
        except OOM_ERR:
            oom = True
        if not oom:
            break
        gc.collect(); free_gpu()
        freeze += 4
        if freeze > 16:
            raise RuntimeError(f"{m['tag']}：顯存不足，已凍結到第 16 層仍然放不下；請改用較小的模型或把 bs 降為 2")
        print(f"  顯存不足 → 改為凍結前 {freeze} 層後重試")
    save_bundle(bundle, path)
    (path / "meta.json").write_text(json.dumps(dict(max_len=max_len, freeze=freeze, swap=bool(m.get("swap")))))
    offload(bundle); del bundle, ex
    free_gpu()


for _m in CLS_MEMBERS:
    if _m["tag"] in ACTIVE_MEMBERS:
        train_member(_m)


def member_logits(m, split, lists, key=""):
    path = member_path(m)
    meta = json.loads((path / "meta.json").read_text()) if (path / "meta.json").exists() else {}
    max_len = meta.get("max_len", m.get("max_len", CLS_MAX_LEN))
    cache = None
    if split["name"] in ("dev", "test"):
        _cf = path / "config.json" if (path / "config.json").exists() else path / "adapter_config.json"   # LLM 只存 LoRA 轉接器
        cache = CACHE_DIR / f"cls_logits_{m['tag']}_{int(_cf.stat().st_mtime)}_{RR_TAG}_{HOP_DELTA}_{split['name']}{key}.npy"
        if cache.exists():
            return np.load(cache)
    pairs = [(split["claim"][qi], ctx_text([(c[1], c[2], c[3]) for c in lists[qi][:CLS_CTX_K]])) for qi in range(len(split["claim"]))]
    if m.get("kind") == "llm":
        bundle = load_llm(path, m["model"])
        lg = score_llm(bundle, [llm_parts(a, b) for a, b in pairs], max_len, m.get("infer_bs", 16), f"分類 {split['name']} / {m['tag']}")
        offload(bundle); del bundle
        free_gpu()
        if cache is not None:
            np.save(cache, lg)
        return lg
    if meta.get("swap"):
        pairs = [(b, a) for a, b in pairs]
    bundle = load_bundle(path)
    lg = score_pairs(bundle, pairs, max_len, m.get("infer_bs", INFER_BS), f"分類 {split['name']} / {m['tag']}")
    offload(bundle); del bundle
    free_gpu()
    if cache is not None:
        np.save(cache, lg)
    return lg


def log_softmax(x):
    x = x - x.max(1, keepdims=True)
    return x - np.log(np.exp(x).sum(1, keepdims=True))

dev 的輸入長度（token）：中位數 253、90 百分位 357；超過 256 而被截斷的比例 = 0.485


In [16]:
# ----- Dev 評估：分類準確率、CFEVER Score -----
MEMBERS = [m for m in CLS_MEMBERS if m["tag"] in ACTIVE_MEMBERS]


def classify(split, final_lists, members=None):
    """集成：各成員 log-softmax 的平均（之後的偏差校準直接加在這個分數上）。"""
    ms = members or MEMBERS
    return np.mean([log_softmax(member_logits(m, split, final_lists)) for m in ms], axis=0)


def predict_with_prior(logits, correct):
    if correct:
        logits = logits + np.log(np.array(PRIOR_TARGET) / np.maximum(cls_prior, 1e-6))
    return logits.argmax(1)


def cfever_score(split, pred, lists):
    hit = 0
    for qi in range(len(split["claim"])):
        if pred[qi] != split["label"][qi]:
            continue
        if split["label"][qi] == 2:
            hit += 1
            continue
        top = {(titles[c[1]], c[2]) for c in lists[qi][:5]}
        hit += any(all(x in top for x in g) for g in split["gold"][qi])
    return hit / len(split["claim"])


dev_logits = classify(dev, EV["dev"])
dev_y = dev["label"]
dev_ev_ok = np.array([lab == 2 or any(all(x in {(titles[c[1]], c[2]) for c in EV_OUT["dev"][qi][:5]} for x in g) for g in dev["gold"][qi])
                      for qi, lab in enumerate(dev_y)])      # NEI 不看證據；S/R 需要前 5 句涵蓋任一組標準證據


def cf(pred, y, ok):
    return float(((pred == y) & ok).mean())


member_acc = {m["tag"]: float((member_logits(m, dev, EV["dev"]).argmax(1) == dev_y).mean()) for m in MEMBERS}
print("各成員 dev accuracy（未校準）：", {k: round(v, 4) for k, v in member_acc.items()}, "| 集成：", round(float((dev_logits.argmax(1) == dev_y).mean()), 4))
acc_plain = float((dev_logits.argmax(1) == dev_y).mean())
acc_prior = float((predict_with_prior(dev_logits, True) == dev_y).mean())   # 舊做法（類別先驗修正），僅作消融
cf_plain = cf(dev_logits.argmax(1), dev_y, dev_ev_ok)
# 集成成員的所有組合（未校準）：看哪些成員真的有貢獻
import itertools
_ml = {m["tag"]: log_softmax(member_logits(m, dev, EV["dev"])) for m in MEMBERS}
subset_rows = []
for r_ in range(1, len(MEMBERS) + 1):
    for comb in itertools.combinations(_ml, r_):
        lg = np.mean([_ml[t] for t in comb], axis=0)
        subset_rows.append(dict(members="+".join(comb), acc=float((lg.argmax(1) == dev_y).mean()), cfever=cf(lg.argmax(1), dev_y, dev_ev_ok)))
subset_df = pd.DataFrame(subset_rows).sort_values("cfever", ascending=False).reset_index(drop=True)
print("成員組合的 dev 表現（未校準）：")
print(subset_df.round(4).to_string(index=False))

# ----- 新成員的配對比較（McNemar）：同一批 dev 題目，看加入新成員前後各翻轉了幾題 -----
# 只看「平均分數差多少」會被雜訊誤導：兩個相近的集成通常有幾百題預測不同，差異的標準誤大約是 sqrt(翻轉題數)/題數。
# 這裡直接印出兩個方向的翻轉題數，差值小於 2 個標準誤就代表「分不出高下」，不該據此換模型。
if NEW_MEMBER in _ml and len(_ml) > 1:
    def _hit(tags):
        p = np.mean([_ml[t] for t in tags], axis=0).argmax(1)
        return (p == dev_y) & dev_ev_ok
    _wo = _hit([t for t in _ml if t != NEW_MEMBER])
    _wi = _hit(list(_ml))
    _n01, _n10 = int((_wo & ~_wi).sum()), int((~_wo & _wi).sum())
    _d = (_n10 - _n01) / len(dev_y)
    _se = math.sqrt(max(_n01 + _n10, 1)) / len(dev_y)
    print(f"\n加入 {NEW_MEMBER} 的配對比較（dev CFEVER）：{_wo.mean():.4f} → {_wi.mean():.4f}（差 {_d:+.4f}）")
    print(f"  變好 {_n10} 題、變壞 {_n01} 題 → 標準誤約 {_se:.4f}；差值 / 標準誤 = {_d / _se:+.2f}")
    print("  |差值| < 2 倍標準誤 = 統計上分不出高下；此時留著新成員也無妨（平均通常不會變差），但別只憑這個數字換掉已確認分數的提交檔。")

# ----- 校準：替 refutes、NEI 加上偏差，直接最大化 dev 的 CFEVER Score（只有 2 個參數）-----
GRID = np.arange(-4, 4.01, 0.25)


def fit_bias(logits, y, ok):
    best = (-1.0, 0.0, 0.0)
    for b1 in GRID:
        for b2 in GRID:
            s = cf((logits + np.array([0.0, b1, b2])).argmax(1), y, ok) - 1e-4 * (abs(b1) + abs(b2))   # 同分時偏好較小的偏差
            if s > best[0]:
                best = (s, b1, b2)
    return np.array([0.0, best[1], best[2]])


perm = np.random.default_rng(SEED).permutation(len(dev_y))
halves = [perm[:len(perm) // 2], perm[len(perm) // 2:]]
cross = []     # 二折交叉驗證：用一半 dev 找偏差、在另一半驗證，估計「校準真正帶來的提升」
for a, b in ((0, 1), (1, 0)):
    bias = fit_bias(dev_logits[halves[a]], dev_y[halves[a]], dev_ev_ok[halves[a]])
    cross.append((cf((dev_logits[halves[b]] + bias).argmax(1), dev_y[halves[b]], dev_ev_ok[halves[b]]),
                  cf(dev_logits[halves[b]].argmax(1), dev_y[halves[b]], dev_ev_ok[halves[b]])))
cf_cross, cf_cross_base = float(np.mean([c[0] for c in cross])), float(np.mean([c[1] for c in cross]))
USE_CAL = cf_cross > cf_cross_base + 0.005          # 交叉驗證至少提升 0.5 個百分點才採用
CAL_BIAS = fit_bias(dev_logits, dev_y, dev_ev_ok) if USE_CAL else np.zeros(3)
print(f"校準：交叉驗證 CFEVER {cf_cross_base:.4f} -> {cf_cross:.4f}；採用校準 = {USE_CAL}；偏差(supports/refutes/NEI) = {CAL_BIAS}")
dev_pred = (dev_logits + CAL_BIAS).argmax(1)
assert abs(cfever_score(dev, dev_pred, EV_OUT["dev"]) - cf(dev_pred, dev_y, dev_ev_ok)) < 1e-9   # 與前面的 CFEVER 公式一致

# 上限分析：若檢索完美（S/R 直接給標準證據），分類器能到多少？
gold_lists = []
for qi in range(len(dev["claim"])):
    if dev["label"][qi] == 2:
        gold_lists.append(EV["dev"][qi])
    else:
        g = min(dev["gold"][qi], key=lambda g: (len({t for t, _ in g}), len(g)))
        items = []
        for t, s in g:
            j = title2idx.get(t)
            items += [(0.0, j, sid, text, ()) for sid, text, _ in (get_sentences(j) if j is not None else ()) if sid == s]
        all_gold = {(t, s) for gg in dev["gold"][qi] for t, s in gg}
        fill = [c for c in EV["dev"][qi] if (titles[c[1]], c[2]) not in all_gold][:max(0, CLS_CTX_K - len(items))]   # 補滿 4 句，與訓練時的輸入長度一致
        gold_lists.append(items + fill)
oracle_logits = classify(dict(claim=dev["claim"], name="dev-oracle"), gold_lists)
acc_oracle = float((oracle_logits.argmax(1) == dev_y).mean())

metrics = dict(
    page_recall_at5=page_recall(dev, ranked["dev"], 5), page_recall_at_topk=page_recall(dev, ranked["dev"], PAGE_TOPK),
    evidence_recall_at5=evidence_recall(dev, EV_OUT["dev"], 5), evidence_recall_at5_reranker=evidence_recall(dev, EV["dev"], 5), use_evidence_selector=bool(SEL_ON), selector_feature_set=SEL_SET, selector_oof_recall=SEL_OOF, use_reranker2=bool(USE_RR2), evidence_recall_at5_no_hop=evidence_recall(dev, rr["dev"][0], 5), hop_delta=HOP_DELTA,
    classification_acc_no_calibration=acc_plain, classification_acc_old_prior_correction=acc_prior,
    classification_acc_calibrated=float((dev_pred == dev_y).mean()), classification_acc_oracle_evidence=acc_oracle,
    active_members=ACTIVE_MEMBERS, member_dev_acc=member_acc, ensemble_subsets=subset_df.round(4).to_dict("records"),
    cfever_score_no_calibration=cf_plain, cfever_score_calibration_crossfit=cf_cross, cfever_score_crossfit_baseline=cf_cross_base,
    use_calibration=bool(USE_CAL), calibration_bias=CAL_BIAS.tolist(), cfever_score=cf(dev_pred, dev_y, dev_ev_ok),
)
per_label = {LABEL_OUT[k]: float((dev_pred[dev_y == k] == k).mean()) for k in range(3)}
print(json.dumps(metrics, indent=2, ensure_ascii=False))
print("各類別 recall（最終預測）：", {k: round(v, 4) for k, v in per_label.items()})
print("混淆矩陣（列=真實，欄=預測；supports/refutes/NEI），未校準：")
print(pd.crosstab(pd.Series(dev_y, name="true"), pd.Series(dev_logits.argmax(1), name="pred")))
if USE_CAL:
    print("校準後：")
    print(pd.crosstab(pd.Series(dev_y, name="true"), pd.Series(dev_pred, name="pred")))

各成員 dev accuracy（未校準）： {'macbert_base': 0.7137, 'roberta_base': 0.7227, 'erlangshen_110m': 0.747, 'erlangshen_330m': 0.7663, 'erlangshen_330m_b': 0.7647, 'qwen06_pilot': 0.7103} | 集成： 0.778
成員組合的 dev 表現（未校準）：
                                                                                 members    acc  cfever
             macbert_base+roberta_base+erlangshen_110m+erlangshen_330m+erlangshen_330m_b 0.7790  0.7397
                          macbert_base+erlangshen_110m+erlangshen_330m+erlangshen_330m_b 0.7813  0.7397
                                            roberta_base+erlangshen_110m+erlangshen_330m 0.7783  0.7383
             macbert_base+erlangshen_110m+erlangshen_330m+erlangshen_330m_b+qwen06_pilot 0.7803  0.7377
macbert_base+roberta_base+erlangshen_110m+erlangshen_330m+erlangshen_330m_b+qwen06_pilot 0.7780  0.7370
                                          roberta_base+erlangshen_110m+erlangshen_330m_b 0.7757  0.7363
                                            macbert_base+erlang

校準：交叉驗證 CFEVER 0.7370 -> 0.7463；採用校準 = True；偏差(supports/refutes/NEI) = [0.  0.5 1. ]


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2481.75it/s]

分類 dev-oracle / macbert_base:   0%|          | 0/47 [00:00<?, ?it/s]

分類 dev-oracle / macbert_base:  64%|██████▍   | 30/47 [00:10<00:05,  2.97it/s]

... (123 outputs left out)

分類 dev-oracle / qwen06_pilot:  94%|█████████▎| 176/188 [02:11<00:10,  1.15it/s]

分類 dev-oracle / qwen06_pilot:  99%|█████████▉| 186/188 [02:12<00:01,  1.08it/s]

分類 dev-oracle / qwen06_pilot: 100%|██████████| 188/188 [02:14<00:00,  1.40it/s]

{
  "page_recall_at5": 0.9175,
  "page_recall_at_topk": 0.9245,
  "evidence_recall_at5": 0.878,
  "evidence_recall_at5_reranker": 0.8615,
  "use_evidence_selector": true,
  "selector_feature_set": "base",
  "selector_oof_recall": {
    "base": {
      "1": 0.497,
      "3": 0.8275,
      "5": 0.878
    },
    "base+link": {
      "1": 0.501,
      "3": 0.8285,
      "5": 0.878
    },
    "base+link+rr2": {
      "1": 0.5105,
      "3": 0.8305,
      "5": 0.878
    }
  },
  "use_reranker2": true,
  "evidence_recall_at5_no_hop": 0.8615,
  "hop_delta": null,
  "classification_acc_no_calibration": 0.778,
  "classification_acc_old_prior_correction": 0.7833333333333333,
  "classification_acc_calibrated": 0.7806666666666666,
  "classification_acc_oracle_evidence": 0.803,
  "active_members": [
    "macbert_base",
    "roberta_base",
    "erlangshen_110m",
    "erlangshen_330m",
    "erlangshen_330m_b",
    "qwen06_pilot"
  ],
  "member_dev_acc": {
    "macbert_base": 0.7136666666666667,
    "r

## 7. 產生 submission.csv

格式與 sample code 相同：`id, predicted_label, predicted_evidence`；預測 NEI 的證據填 `[]`，其餘填前 5 句 `[page_id, sentence_id]`。

In [17]:
test_logits = classify(test, EV["test"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def make_submission(pred, path):
    evs = ["[]" if p == 2 else json.dumps([[titles[c[1]], c[2]] for c in EV_OUT["test"][qi][:5]], ensure_ascii=False) for qi, p in enumerate(pred)]
    out = pd.DataFrame({"id": test["id"], "predicted_label": [LABEL_OUT[p] for p in pred], "predicted_evidence": evs})
    out.to_csv(path, index=False, encoding="utf-8")
    return out


_sfx = "_quick" if QUICK_TEST else ""
sub_path = OUTPUT_DIR / f"submission{_sfx}.csv"                 # 正式版（若採用校準，則含校準）
submission = make_submission((test_logits + CAL_BIAS).argmax(1), sub_path)
sub_nocal = make_submission(test_logits.argmax(1), OUTPUT_DIR / f"submission_nocal{_sfx}.csv")   # 未校準版，可上傳 Kaggle 做 A/B 比較
(OUTPUT_DIR / ("metrics_quick.json" if QUICK_TEST else "metrics.json")).write_text(
    json.dumps(dict(metrics, per_label_recall=per_label, page_table=page_df.round(4).to_dict(), evidence_table=ev_df.round(4).to_dict()),
               indent=2, ensure_ascii=False), encoding="utf-8")
print("輸出筆數：", len(submission), "| 輸出位置：", sub_path.resolve())
print("預測標籤分佈（正式版）：", submission["predicted_label"].value_counts().to_dict())
print("預測標籤分佈（未校準）：", sub_nocal["predicted_label"].value_counts().to_dict())
# 額外分析：test 內有 3 種來源明顯不同的題目（見 Report 4(d)）：
#   A. id<=31996：與 train/dev 的 id 交錯，分佈類似 dev；
#   B. id>31996 且含「最喜歡」（1000 題）：虛構的個人偏好句，train/dev 完全沒有這種模板；
#   C. id>31996 其餘（2000 題）：長句、整句像是直接抄自維基。
_ids = np.array(test["id"])
_fav = np.array(["最喜歡" in c for c in test["claim"]])
groups = {"A id<=31996": _ids <= 31996, "B id>31996 含『最喜歡』": (_ids > 31996) & _fav, "C id>31996 其他": (_ids > 31996) & ~_fav}
for _name, _m in groups.items():
    print(f"{_name:24s} n={int(_m.sum()):4d} 預測分佈：", submission.loc[_m, "predicted_label"].value_counts(normalize=True).round(3).to_dict())
submission.head()

輸出筆數： 6000 | 輸出位置： /home/yiting/NLP-Assignment3/output/final/submission.csv
預測標籤分佈（正式版）： {'NOT ENOUGH INFO': 2159, 'supports': 1941, 'refutes': 1900}
預測標籤分佈（未校準）： {'supports': 2081, 'NOT ENOUGH INFO': 1994, 'refutes': 1925}
A id<=31996              n=3000 預測分佈： {'NOT ENOUGH INFO': 0.38, 'supports': 0.313, 'refutes': 0.307}
B id>31996 含『最喜歡』        n=1000 預測分佈： {'NOT ENOUGH INFO': 1.0}
C id>31996 其他            n=2000 預測分佈： {'supports': 0.501, 'refutes': 0.49, 'NOT ENOUGH INFO': 0.009}


,id,predicted_label,predicted_evidence
0,29579,NOT ENOUGH INFO,[]
1,3895,supports,"[[""鄱陽湖"", 0], [""鄱陽湖"", 2], [""江西省"", 1], [""鄱陽湖"", 1..."
2,24522,refutes,"[[""赫伯特·胡佛"", 1], [""胡佛"", 5], [""赫伯特·胡佛"", 0], [""胡佛..."
3,11060,NOT ENOUGH INFO,[]
4,23607,supports,"[[""馬肯森級大巡洋艦"", 7], [""馬肯森級大巡洋艦"", 0], [""馬肯森級大巡洋艦""..."


In [18]:
# ----- NEI 偏差掃描：test 各組的 NEI 比例 + dev CFEVER；另存備選提交檔 -----
# 檔名含「成員代號」與偏差，重跑也不會覆蓋別的組合的檔案；每份提交請記錄「檔名 → public 分數」，最後只能選 2 份。
import hashlib
_mtag = f"m{len(ACTIVE_MEMBERS)}_" + hashlib.md5("|".join(ACTIVE_MEMBERS).encode()).hexdigest()[:6]
_mtag += "_sel" if SEL_ON else ""                  # 有用證據選擇器的提交檔另外命名，避免覆蓋舊檔
print(f"提交檔前綴 {_mtag} = {ACTIVE_MEMBERS}")
make_submission((test_logits + CAL_BIAS).argmax(1), OUTPUT_DIR / f"submission_{_mtag}_cal{_sfx}.csv")      # = submission.csv（含 dev 校準）
for _b in (0.0, 0.5, 1.0, 1.5, 2.0, 2.5):
    _pred = (test_logits + np.array([0.0, 0.0, _b])).argmax(1)
    _share = {g: round(float((_pred[m] == 2).mean()), 3) for g, m in groups.items()}
    _dcf = cf((dev_logits + np.array([0.0, 0.0, _b])).argmax(1), dev_y, dev_ev_ok)
    print(f"NEI偏差 {_b}: test 各組 NEI 比例 {_share} | 全部 {float((_pred == 2).mean()):.3f} | dev CFEVER {_dcf:.4f}")
    make_submission(_pred, OUTPUT_DIR / f"submission_{_mtag}_nei{_b}{_sfx}.csv")
print("\nv7 的提交檔（public：nei1.5 → 0.881、nei2.0 → 0.876）；最終提交改由第 9 節產生。")

提交檔前綴 m6_037f63_sel = ['macbert_base', 'roberta_base', 'erlangshen_110m', 'erlangshen_330m', 'erlangshen_330m_b', 'qwen06_pilot']
NEI偏差 0.0: test 各組 NEI 比例 {'A id<=31996': 0.326, 'B id>31996 含『最喜歡』': 1.0, 'C id>31996 其他': 0.007} | 全部 0.332 | dev CFEVER 0.7370
NEI偏差 0.5: test 各組 NEI 比例 {'A id<=31996': 0.368, 'B id>31996 含『最喜歡』': 1.0, 'C id>31996 其他': 0.009} | 全部 0.353 | dev CFEVER 0.7390
NEI偏差 1.0: test 各組 NEI 比例 {'A id<=31996': 0.415, 'B id>31996 含『最喜歡』': 1.0, 'C id>31996 其他': 0.009} | 全部 0.377 | dev CFEVER 0.7387


NEI偏差 1.5: test 各組 NEI 比例 {'A id<=31996': 0.488, 'B id>31996 含『最喜歡』': 1.0, 'C id>31996 其他': 0.01} | 全部 0.414 | dev CFEVER 0.7277
NEI偏差 2.0: test 各組 NEI 比例 {'A id<=31996': 0.56, 'B id>31996 含『最喜歡』': 1.0, 'C id>31996 其他': 0.011} | 全部 0.451 | dev CFEVER 0.6957


NEI偏差 2.5: test 各組 NEI 比例 {'A id<=31996': 0.628, 'B id>31996 含『最喜歡』': 1.0, 'C id>31996 其他': 0.021} | 全部 0.488 | dev CFEVER 0.6550

v7 的提交檔（public：nei1.5 → 0.881、nei2.0 → 0.876）；最終提交改由第 9 節產生。


## 8. v8：讓分類器讀「選擇器挑出的證據」+ 交叉驗證學集成權重 + refutes 偏差（實驗，三項都沒有採用）

**這一節是負面結果的紀錄（對應 Report Q4(c)）：三項改動在 dev 或 public 上都沒有明確好處，最終提交仍是第 7 節的 NEI 偏差 1.5 / 2.0。**

1. **證據一致性**：前面的分類器讀的是重排器的前 4 句，但送出的證據是選擇器的前 5 句。這裡讓每個成員改讀**選擇器排序的前 4 句**
   （dev 用 5 折 OOF 的選擇結果，不偷看標準答案；test 用以全部 dev 訓練的選擇器）。不需重訓，只要重新推論。
2. **集成權重**：等權平均時，較弱的成員（qwen、macbert）與最強的 330M 同票。這裡在 dev 上用 5 折交叉驗證學權重
   （以 negative log-likelihood 為目標、權重非負且加總為 1），每一折的權重都只用其他 4 折學，因此 dev 上的數字是 out-of-fold 的。
3. **refutes 偏差**：v7 的 dev 校準結果是 refutes +0.5、NEI +1.0（交叉驗證有效）。NEI 偏差已依 public 定為 1.5，這裡用 dev 檢查 refutes 偏差是否仍有幫助。

所有比較都用**配對檢定**（同一批 dev 題目，看變好/變壞各幾題）；只有在部署用的 NEI 偏差（1.5）下仍然不變差，才採用。

In [19]:
from scipy.optimize import minimize

V8_NEI = 1.5                                    # 部署用的 NEI 偏差（依 public 曲線選定）
V8_INPUTS = {"rr": EV, "sel": EV_OUT}            # 分類器輸入：重排器前 4 句 / 選擇器前 4 句
LS8 = {k: {s["name"]: {m["tag"]: log_softmax(member_logits(m, s, V8_INPUTS[k][s["name"]], key="" if k == "rr" else "_sel"))
                       for m in MEMBERS} for s in (dev, test)} for k in V8_INPUTS}
_tags = [m["tag"] for m in MEMBERS]


def _stack(L, tags):
    return np.stack([L[t] for t in tags])        # (M, n, 3)


def fit_weights(Ls, y):
    """學集成權重：w = softmax(theta)；最小化 dev 的 NLL（加一點 L2，讓權重不要離等權太遠）。"""
    M = Ls.shape[0]

    def nll(theta):
        w = np.exp(theta - theta.max()); w /= w.sum()
        z = log_softmax(np.tensordot(w, Ls, axes=1))
        return -z[np.arange(len(y)), y].mean() + 1e-3 * float((theta ** 2).sum())
    th = minimize(nll, np.zeros(M), method="L-BFGS-B").x
    w = np.exp(th - th.max())
    return w / w.sum()


_order = np.argsort(np.array(dev["id"]))         # 依 id 排序後切成連續 5 折（相鄰 id 常共用頁面，連續切比較保守）
_folds8 = np.array_split(_order, 5)
V8 = {}
for k in V8_INPUTS:
    Ls = _stack(LS8[k]["dev"], _tags)
    V8[(k, "equal")] = Ls.mean(0)
    oof = np.zeros_like(V8[(k, "equal")])
    for f in range(5):
        va = _folds8[f]
        tr = np.concatenate([_folds8[g] for g in range(5) if g != f])
        w = fit_weights(Ls[:, tr], dev_y[tr])
        oof[va] = np.tensordot(w, Ls[:, va], axes=1)
    V8[(k, "cvw")] = oof
W8_FULL = {k: fit_weights(_stack(LS8[k]["dev"], _tags), dev_y) for k in V8_INPUTS}
print("全部 dev 學到的集成權重：")
for k, w in W8_FULL.items():
    print(f"  {k:4s}", {t: round(float(x), 3) for t, x in zip(_tags, w)})


def _hits(lg, bias):
    return ((lg + bias).argmax(1) == dev_y) & dev_ev_ok


def paired(lg_a, lg_b, bias):
    a, b = _hits(lg_a, bias), _hits(lg_b, bias)
    n01, n10 = int((a & ~b).sum()), int((~a & b).sum())
    d = (n10 - n01) / len(dev_y)
    se = math.sqrt(max(n01 + n10, 1)) / len(dev_y)
    return a.mean(), b.mean(), n10, n01, d / se


_base = V8[("rr", "equal")]                     # = v7 的做法
print(f"\nDev CFEVER（out-of-fold），對照組 = 重排器輸入 + 等權（v7）")
_rows8 = []
for (k, wmode), lg in V8.items():
    r = {"輸入": k, "權重": wmode}
    for b in (0.0, 0.5, 1.0, V8_NEI):
        r[f"NEI {b}"] = round(cf((lg + np.array([0, 0, b])).argmax(1), dev_y, dev_ev_ok), 4)
    if (k, wmode) != ("rr", "equal"):
        for b in (0.0, V8_NEI):
            _, _, n10, n01, z = paired(_base, lg, np.array([0, 0, b]))
            r[f"配對@{b}（好/壞, z）"] = f"+{n10}/-{n01}, {z:+.2f}"
    _rows8.append(r)
print(pd.DataFrame(_rows8).to_string(index=False))

# 採用規則：在 NEI 0 與部署偏差 1.5 兩種設定下，配對差值都 >= 0，才算「沒有變差」而採用；兩者都通過時取 z 總和較大者
_cand = []
for key in V8:
    if key == ("rr", "equal"):
        continue
    z0 = paired(_base, V8[key], np.array([0, 0, 0.0]))[4]
    z1 = paired(_base, V8[key], np.array([0, 0, V8_NEI]))[4]
    if z0 >= 0 and z1 >= 0:
        _cand.append((z0 + z1, key))
V8_CHOICE = max(_cand)[1] if _cand else ("rr", "equal")
print("\n採用的 v8 設定（輸入, 權重）：", V8_CHOICE, "" if _cand else "（沒有任何設定在兩種偏差下都不變差 → 維持 v7）")

# refutes 偏差：NEI 固定為 1.5，以二折交叉驗證選 refutes 偏差（0 / 0.25 / 0.5 / 0.75）
_lgC = V8[V8_CHOICE]
_R_GRID = (0.0, 0.25, 0.5, 0.75)
_halves = [_order[: len(_order) // 2], _order[len(_order) // 2:]]
_xf = []
for a, b in ((0, 1), (1, 0)):
    ia, ib = _halves[a], _halves[b]
    r_best = max(_R_GRID, key=lambda r: (cf((_lgC[ia] + np.array([0, r, V8_NEI])).argmax(1), dev_y[ia], dev_ev_ok[ia]), -r))
    _xf.append((cf((_lgC[ib] + np.array([0, r_best, V8_NEI])).argmax(1), dev_y[ib], dev_ev_ok[ib]),
                cf((_lgC[ib] + np.array([0, 0, V8_NEI])).argmax(1), dev_y[ib], dev_ev_ok[ib]), r_best))
print("refutes 偏差（NEI 固定 1.5）各值的 dev CFEVER：",
      {r: round(cf((_lgC + np.array([0, r, V8_NEI])).argmax(1), dev_y, dev_ev_ok), 4) for r in _R_GRID})
print("  二折交叉驗證：", [f"選 {x[2]} → {x[0]:.4f}（不加：{x[1]:.4f}）" for x in _xf])
V8_REF = 0.5 if np.mean([x[0] for x in _xf]) > np.mean([x[1] for x in _xf]) else 0.0
print("  refutes 偏差交叉驗證有提升 → 另外產生 refutes +0.5 的檔案" if V8_REF else "  交叉驗證沒有提升 → 不加 refutes 偏差")

# ----- 產生 v8 提交檔（只在有改變時才產生，避免與 v7 重複上傳）-----
_k, _wm = V8_CHOICE
_Lte = _stack(LS8[_k]["test"], _tags)
test_logits_v8 = _Lte.mean(0) if _wm == "equal" else np.tensordot(W8_FULL[_k], _Lte, axes=1)
_tag8 = f"v8_{_k}_{_wm}"
if V8_CHOICE != ("rr", "equal"):
    for _b in (1.5, 2.0):
        make_submission((test_logits_v8 + np.array([0, 0, _b])).argmax(1), OUTPUT_DIR / f"submission_{_tag8}_nei{_b}{_sfx}.csv")
if V8_REF:
    make_submission((test_logits_v8 + np.array([0, V8_REF, V8_NEI])).argmax(1), OUTPUT_DIR / f"submission_{_tag8}_ref{V8_REF}_nei{V8_NEI}{_sfx}.csv")
for _p in sorted(OUTPUT_DIR.glob(f"submission_v8_*{_sfx}.csv")):
    _d = pd.read_csv(_p)["predicted_label"].value_counts().to_dict()
    print("已產生：", _p.name, _d)

全部 dev 學到的集成權重：
  rr   {'macbert_base': 0.122, 'roberta_base': 0.092, 'erlangshen_110m': 0.142, 'erlangshen_330m': 0.175, 'erlangshen_330m_b': 0.184, 'qwen06_pilot': 0.285}
  sel  {'macbert_base': 0.132, 'roberta_base': 0.092, 'erlangshen_110m': 0.14, 'erlangshen_330m': 0.178, 'erlangshen_330m_b': 0.181, 'qwen06_pilot': 0.276}

Dev CFEVER（out-of-fold），對照組 = 重排器輸入 + 等權（v7）
 輸入    權重  NEI 0.0  NEI 0.5  NEI 1.0  NEI 1.5 配對@0.0（好/壞, z） 配對@1.5（好/壞, z）
 rr equal   0.7370   0.7390   0.7387   0.7277            NaN            NaN
 rr   cvw   0.7350   0.7410   0.7387   0.7300 +22/-28, -0.85 +19/-12, +1.26
sel equal   0.7360   0.7420   0.7423   0.7290 +55/-58, -0.28 +55/-51, +0.39
sel   cvw   0.7347   0.7423   0.7423   0.7310 +61/-68, -0.62 +61/-51, +0.94

採用的 v8 設定（輸入, 權重）： ('rr', 'equal') （沒有任何設定在兩種偏差下都不變差 → 維持 v7）
refutes 偏差（NEI 固定 1.5）各值的 dev CFEVER： {0.0: 0.7277, 0.25: 0.734, 0.5: 0.7393, 0.75: 0.743}
  二折交叉驗證： ['選 0.75 → 0.7813（不加：0.7707）', '選 0.75 → 0.7047（不加：0.6847）']
  refutes 偏差交叉驗證有提升 

## 9. v9：更強、更多樣的集成成員（最終提交來源）

v9 的出發點是錯誤分析：dev 上「分類錯誤」造成的損失（約 22 個百分點）遠大於「證據不完整」（約 4–6 個百分點），所以主要投資在分類器。
另一個關鍵觀察：**凡是在 dev 上擬合的事後調整（NEI / refutes 偏差、校準、堆疊）都不太能轉移到 test**（見第 7、8 節與 9f），
歷史上 dev 與 public 同時進步的只有「更強、更多樣的基礎模型」。因此 v9 只加入「用 train（＋一半 dev）訓練出來的新成員」，集成方式不變（log-softmax 等權平均）。

**採用規則（事先訂好）**：在部署時的 dev NEI 預測比例（0.42 / 0.46 / 0.50）下做配對檢定，三個比例都不變差、且至少一個 z ≥ 2 才採用；public 不能比 v7 低超過雜訊（約 0.004）。

1. **群集感知的證據句選擇器（9a，實驗）**：CFEVER 的 claim 是成群產生的——同一個來源句子會被改寫成好幾個 claim（supports / refutes / NEI 都有），而且 id 相鄰
   （dev 的 2000 題 supports / refutes 中，有 1970 題和 id 相差 ≤ 20 的另一題共用標準證據頁面）。替選擇器加入「鄰居」特徵後，
   依 id 連續切 5 折的 OOF Recall@5 從 0.875 → 0.879，但 public 沒有可見差異（0.877 → 0.878），證據不足，**最終仍使用 v7 的選擇器**。
2. **本機訓練的新成員（9b）**：
   - `qwen06`：Qwen3-0.6B-Base + LoRA，用**全部**訓練資料訓練 2 epoch（v4 的 pilot 只用 6000 筆、1 epoch）：dev accuracy 0.710 → 0.763，取代 pilot。
   - `qwen06_sib`：同上，但輸入在證據後面附上 2 句「兄弟 claim」（同一切分中 id 相差 ≤ 20、字元二元組 Jaccard ≥ 0.1 的最相似 claim；只用 claim 文字與 id，不用標籤）。
     同一群的 claim 是同一個來源句的不同改寫，模型可以學到「與兄弟相比多了什麼資訊」這類線索：dev accuracy 0.775，是單一模型最高。
3. **在 Kaggle GPU（16GB）上訓練的成員（9d）**：完整微調（本機 4GB 只能凍結一半的層）、不同的模型家族（XLM-R-large-XNLI、mDeBERTa-v3-XNLI）與兄弟 claim 輸入。
4. **NEI 偏差的選法**：換了集成之後同一個偏差值代表的意義會變，所以改成**對齊 test 的 NEI 預測比例**：選偏差讓集成在 test 上被判成 NEI 的比例，
   等於 v7 在某個偏差下的比例（只用 test 的「預測分佈」，不用任何 test 標籤）。v7 的最佳點是偏差 1.5；12 成員的 v9 集成在 public 上的掃描是
   0.884 / 0.883 / **0.889** / 0.888 / 0.888 / 0.882（對齊 v7 偏差 1.25 / 1.5 / 1.75 / 1.875 / 2.0 / 2.25），更強的集成在 1.75～2.0 形成平台、2.25 開始下降，
   所以最終兩份提交取平台的兩端：**對齊 1.75 與 2.0**（A 組被判成 NEI 的比例 52.3% / 55.6%）。

In [20]:
# ===== 9a. 群集感知的證據句選擇器（sibling features；實驗，最終未採用）=====
SIB_GAP, SIB_JAC, SIB_WIN = 20, 0.10, 12          # 鄰居：id 相差 <= 20 且 claim 字元二元組 Jaccard >= 0.10；只看 id 排序後前後各 12 題
SEL_BASE_COLS = SEL_SETS["base"]                    # v5 以來使用的 21 個 base 特徵


def sib_neighbours(split):
    ids = np.array(split["id"]); order = np.argsort(ids, kind="stable")
    B = [_bg(c) for c in split["claim"]]
    nb = [[] for _ in ids]
    for pos, qi in enumerate(order):
        for pj in range(max(0, pos - SIB_WIN), min(len(order), pos + SIB_WIN + 1)):
            qj = order[pj]
            if qj == qi or abs(int(ids[qj]) - int(ids[qi])) > SIB_GAP:
                continue
            jac = len(B[qi] & B[qj]) / max(1, len(B[qi] | B[qj]))
            if jac >= SIB_JAC:
                nb[qi].append(qj)
    return nb


def sib_features(lists, nb, meta):
    """每個候選句 (qi, k)：在鄰居前 SEL_N 名中的最高重排分數、出現在鄰居前 5 名 / 第 1 名的次數、鄰居數、比例。"""
    top = [{(c[1], c[2]): (r, c[0]) for r, c in enumerate(l[:SEL_N])} for l in lists]
    rows = []
    for qi, k in meta:
        c = lists[qi][k]
        hit = [top[j].get((c[1], c[2])) for j in nb[qi]]
        hit = [h for h in hit if h is not None]
        n5 = sum(h[0] < 5 for h in hit)
        rows.append([max([h[1] for h in hit], default=-10.0), n5, sum(h[0] == 0 for h in hit), len(nb[qi]), n5 / len(nb[qi]) if nb[qi] else 0.0])
    return np.array(rows, dtype=np.float32)


_blk = np.zeros(len(dev["id"]), dtype=int)                       # 依 id 連續切 5 折：同一群的兄弟題一定在同一折
for _f, _ix in enumerate(np.array_split(np.argsort(np.array(dev["id"]), kind="stable"), 5)):
    _blk[_ix] = _f
_fold_of_row = _blk[q_dev]


def sel_oof(X):
    oof = np.zeros(len(y_dev))
    for f in range(5):
        tr, va = np.where((_fold_of_row != f) & sr_mask)[0], np.where(_fold_of_row == f)[0]
        oof[va] = sel_model().fit(X[tr], y_dev[tr]).predict_proba(X[va])[:, 1]
    return oof


NB9 = {"dev": sib_neighbours(dev), "test": sib_neighbours(test)}
X9_dev = np.hstack([X_dev[:, SEL_BASE_COLS], sib_features(EV["dev"], NB9["dev"], meta_dev)])
X9_te = np.hstack([X_te[:, SEL_BASE_COLS], sib_features(EV["test"], NB9["test"], meta_te)])
SEL7_BLOCK = sel_reorder(EV["dev"], meta_dev, sel_oof(X_dev[:, SEL_BASE_COLS]))     # v5/v7 的選擇器，但改用嚴格的連續 id 折
SEL9_DEV = sel_reorder(EV["dev"], meta_dev, sel_oof(X9_dev))
print("平均鄰居數：dev", round(float(np.mean([len(x) for x in NB9["dev"]])), 2), "| test", round(float(np.mean([len(x) for x in NB9["test"]])), 2))
print("證據 Recall@1/3/5（dev，依 id 連續切 5 折的 OOF）")
print("  重排器                 ", [round(evidence_recall(dev, EV["dev"], K), 4) for K in (1, 3, 5)])
print("  選擇器 base（v7）      ", [round(evidence_recall(dev, SEL7_BLOCK, K), 4) for K in (1, 3, 5)])
print("  選擇器 base + 鄰居（v9）", [round(evidence_recall(dev, SEL9_DEV, K), 4) for K in (1, 3, 5)])
SIB_ON = evidence_recall(dev, SEL9_DEV, 5) >= evidence_recall(dev, SEL7_BLOCK, 5) + 0.002   # 至少多 0.2 個百分點才採用
if SIB_ON:
    _sel9 = sel_model().fit(X9_dev[sr_mask], y_dev[sr_mask])
    EV_OUT9 = {"dev": SEL9_DEV, "test": sel_reorder(EV["test"], meta_te, _sel9.predict_proba(X9_te)[:, 1])}
else:
    EV_OUT9 = {"dev": SEL7_BLOCK, "test": EV_OUT["test"]}
print("採用鄰居特徵：", SIB_ON)
_g9 = np.where(np.array(test["id"]) <= 31996, "A", np.where(np.array(["最喜歡" in c for c in test["claim"]]), "B", "C"))
for _g in "ABC":                                                  # C 組（抄自維基的長句）幾乎全對，確認新選擇器沒有把原本的第 1 名擠出前 5 句
    _m = np.where(_g9 == _g)[0]
    _kept = np.mean([(EV_OUT["test"][i][0][1], EV_OUT["test"][i][0][2]) in {(c[1], c[2]) for c in EV_OUT9["test"][i][:5]} for i in _m])
    print(f"  test {_g} 組：v7 選擇器的第 1 句仍在 v9 前 5 句的比例 = {_kept:.4f}")
dev_ev_ok9 = np.array([lab == 2 or any(all(x in {(titles[c[1]], c[2]) for c in EV_OUT9["dev"][qi][:5]} for x in g) for g in dev["gold"][qi])
                       for qi, lab in enumerate(dev_y)])
dev_ev_ok7b = np.array([lab == 2 or any(all(x in {(titles[c[1]], c[2]) for c in SEL7_BLOCK[qi][:5]} for x in g) for g in dev["gold"][qi])
                        for qi, lab in enumerate(dev_y)])

平均鄰居數：dev 5.09 | test 3.46
證據 Recall@1/3/5（dev，依 id 連續切 5 折的 OOF）
  重排器                  [0.4955, 0.8005, 0.8615]
  選擇器 base（v7）       [0.4935, 0.8275, 0.875]
  選擇器 base + 鄰居（v9） [0.4995, 0.8335, 0.879]


採用鄰居特徵： True
  test A 組：v7 選擇器的第 1 句仍在 v9 前 5 句的比例 = 1.0000
  test B 組：v7 選擇器的第 1 句仍在 v9 前 5 句的比例 = 0.9970
  test C 組：v7 選擇器的第 1 句仍在 v9 前 5 句的比例 = 1.0000


In [21]:
# ===== 9b. 本機訓練的新成員：完整版 Qwen3-0.6B、附兄弟 claim 的 Qwen3-0.6B =====
# qwen06（設定在第 1 節的 CLS_MEMBERS）：全部訓練資料、2 epoch，RTX 3050 約 4.5 小時；訓練完會存檔，之後重跑直接載入。
QWEN_FULL = next(m for m in CLS_MEMBERS if m["tag"] == "qwen06")
train_member(QWEN_FULL)
SIB_N, SIB_GAP_ID, SIB_MIN_JAC = 2, 20, 0.10
QWEN_SIB_LEN, QWEN_SIB_SEED = 448, 41


def sibling_claims(split, n=SIB_N):
    """同一切分中 id 相差 <= 20、字元二元組 Jaccard >= 0.1 的 claim，依 Jaccard 由高到低取 n 句（不使用標籤）。"""
    ids = np.array(split["id"]); order = np.argsort(ids, kind="stable")
    B = [_bg(c) for c in split["claim"]]
    out = [[] for _ in ids]
    for pos, qi in enumerate(order):
        cand = []
        for pj in range(max(0, pos - 12), min(len(order), pos + 13)):
            qj = order[pj]
            if qj != qi and abs(int(ids[qj]) - int(ids[qi])) <= SIB_GAP_ID:
                jac = len(B[qi] & B[qj]) / max(1, len(B[qi] | B[qj]))
                if jac >= SIB_MIN_JAC:
                    cand.append((jac, qj))
        cand.sort(key=lambda t: (-t[0], t[1]))
        out[qi] = [split["claim"][qj] for _, qj in cand[:n]]
    return out


def with_sibs(ctx, sibs):
    return ctx + (" [SEP] 相關敘述：" + "；".join(sibs) if sibs else "")


SIBS = {s["name"]: sibling_claims(s) for s in (train, dev, test)}
print("有兄弟 claim 的比例：", {k: round(float(np.mean([len(x) > 0 for x in v])), 3) for k, v in SIBS.items()})


def qwen_sib_parts(split):
    return [llm_parts(split["claim"][qi], with_sibs(ctx_text([(c[1], c[2], c[3]) for c in EV[split["name"]][qi][:CLS_CTX_K]]), SIBS[split["name"]][qi]))
            for qi in range(len(split["claim"]))]


def qwen_sib_member():
    """附兄弟 claim 的 Qwen3-0.6B + LoRA（只用 train 訓練）；回傳 {(split, "rr"): log-softmax}。"""
    path = CACHE_DIR / "cls_qwen06_sib"
    files = {s: CACHE_DIR / f"cls_logits_qwen06_sib_{s}.npy" for s in ("dev", "test")}
    if not all(p.exists() for p in files.values()):
        if not path.exists():
            first_sibs = {}
            for qi, c in enumerate(train["claim"]):
                first_sibs.setdefault(c, SIBS["train"][qi])
            ex = [(c, with_sibs(ctx, first_sibs.get(c, [])), lab) for c, ctx, lab in build_cls_examples(seed=QWEN_SIB_SEED)]
            bundle = train_llm_clf(ex, "Qwen/Qwen3-0.6B-Base", QWEN_SIB_LEN, 2, 1e-4, 4, 4, "訓練 qwen06_sib", seed=QWEN_SIB_SEED)
            save_llm(bundle, path); offload(bundle); del bundle, ex
            free_gpu()
        bundle = load_llm(path, "Qwen/Qwen3-0.6B-Base")
        for s in (dev, test):
            np.save(files[s["name"]], score_llm(bundle, qwen_sib_parts(s), QWEN_SIB_LEN, 16, f"分類 {s['name']} / qwen06_sib"))
        offload(bundle); del bundle
        free_gpu()
    return {(s, "rr"): log_softmax(np.load(files[s])) for s in ("dev", "test")}


LOCAL9 = {"qwen06": {(s["name"], "rr"): log_softmax(member_logits(QWEN_FULL, s, EV[s["name"]])) for s in (dev, test)},
          "qwen06_sib": qwen_sib_member()}
for _t, _r in LOCAL9.items():
    print(f"本機新成員 {_t}：dev accuracy {(_r[('dev', 'rr')].argmax(1) == dev_y).mean():.4f}")

有兄弟 claim 的比例： {'train': 0.996, 'dev': 0.995, 'test': 0.726}
本機新成員 qwen06：dev accuracy 0.7630
本機新成員 qwen06_sib：dev accuracy 0.7753


In [22]:
# ===== 9d. 在 Kaggle Notebook GPU（16GB）上訓練的成員 =====
# 本機 RTX 3050 只有 4GB，330M 模型必須凍結前 8 層、輸入只能放 4 句；下列成員改在 Kaggle 的 GPU（作業說明提到每週 30 小時）訓練：
#   - 完整微調（不凍結）、最大長度 512；不同模型家族（XLM-R-large-XNLI、mDeBERTa-v3-XNLI）；多數成員在輸入中附上 2～3 句「兄弟 claim」（與 9b 相同的定義，不用標籤）。
#   - 訓練資料與推論輸入由 export_for_kaggle() 從本 notebook 的重排結果匯出（與 build_cls_examples 的建構方式相同），
#     訓練程式碼就是下面的 KG_SCRIPT（上傳成 Kaggle 私人 dataset + 私人 notebook 執行），輸出 dev / test 兩種視角的 logits，下載到 output/cache/kaggle/<tag>/。
#   - 本機重跑時直接讀取下載的 logits；若本機有 >= 15GB 顯存的 GPU，也可以把 KG_RUN_LOCAL 設成 True 直接在本機訓練。
KG_DIR = CACHE_DIR / "kaggle"
KG_DATA = KG_DIR / "data"
KG_RUN_LOCAL = False
KG_SPECS = [
    dict(tag="xlmr_xnli_k6", model="joeddav/xlm-roberta-large-xnli", K=6, max_len=512, lr=8e-6, epochs=2, bs=8, accum=2, seed=11,
         swap=True, head_perm=[2, 0, 1], cf=2),
    dict(tag="e330_sib2_k4", model="IDEA-CCNL/Erlangshen-Roberta-330M-NLI", K=4, sib=2, max_len=512, lr=1e-5, epochs=2, bs=16, accum=1, seed=13,
         swap=True, head_perm=[2, 0, 1], cf=0),
    dict(tag="xlmr_sib2_k4", model="joeddav/xlm-roberta-large-xnli", K=4, sib=2, max_len=512, lr=8e-6, epochs=2, bs=8, accum=2, seed=19,
         swap=True, head_perm=[2, 0, 1], cf=0),
    dict(tag="e330_sib3_k6", model="IDEA-CCNL/Erlangshen-Roberta-330M-NLI", K=6, sib=3, max_len=512, lr=1e-5, epochs=2, bs=16, accum=1, seed=17,
         swap=True, head_perm=[2, 0, 1], cf=0),
    dict(tag="mdeberta_sib2_k4", model="MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7", K=4, sib=2, max_len=512, lr=2e-5, epochs=2, bs=16,
         accum=1, seed=23, swap=True, head_perm=[0, 2, 1], cf=0),
]


def export_for_kaggle(topn=10):
    """匯出 train / dev / test 的 claim、重排器前 topn 句（含分數）、選擇器前 topn 句、標準證據句（含文字）。"""
    KG_DATA.mkdir(parents=True, exist_ok=True)
    L = lambda lst: [[disp(c[1]), int(c[2]), round(float(c[0]), 4), c[3], titles[c[1]]] for c in lst[:topn]]

    def gold_groups(groups):
        out = []
        for gr in groups:
            items = []
            for t, s in gr:
                j = title2idx.get(t)
                txt = next((x for sid, x, _ in (get_sentences(j) if j is not None else ()) if sid == s), None)
                items.append([disp(j) if j is not None else t, int(s), txt, t])
            out.append(items)
        return out
    lists = {}
    nei_ids = [i for i in range(len(train["claim"])) if train["label"][i] == 2]
    sr_ids = [i for i in range(len(train["claim"])) if train["label"][i] != 2]
    for ids, tag in ((nei_ids, "NEI"), (sr_ids, "SR")):
        first, _ = cached(CACHE_DIR / f"rerank_{_sig}_{RR_TAG}_train{tag}_{len(ids)}.pkl", lambda: None)
        lists.update(zip(ids, first))
    with open(KG_DATA / "train.jsonl", "w", encoding="utf-8") as f:
        for qi in range(len(train["claim"])):
            f.write(json.dumps(dict(id=int(train["id"][qi]), claim=train["claim"][qi], label=int(train["label"][qi]), rr=L(lists[qi]),
                                    gold=gold_groups(train["gold"][qi])), ensure_ascii=False) + "\n")
    with open(KG_DATA / "dev.jsonl", "w", encoding="utf-8") as f:
        for qi in range(len(dev["claim"])):
            f.write(json.dumps(dict(id=int(dev["id"][qi]), claim=dev["claim"][qi], label=int(dev["label"][qi]), rr=L(rr["dev"][0][qi]),
                                    sel=L(EV_OUT["dev"][qi]), gold=gold_groups(dev["gold"][qi])), ensure_ascii=False) + "\n")
    with open(KG_DATA / "test.jsonl", "w", encoding="utf-8") as f:
        for qi in range(len(test["claim"])):
            f.write(json.dumps(dict(id=int(test["id"][qi]), claim=test["claim"][qi], rr=L(rr["test"][0][qi]), sel=L(EV_OUT["test"][qi])),
                               ensure_ascii=False) + "\n")


KG_SCRIPT = r'''# CFEVER classifier member training on a Kaggle GPU (same example construction as build_cls_examples in the notebook).
# Generative AI usage: written with the help of Claude (Anthropic); see the notebook's GenAI statement.
import os, json, math, random, time, glob
import numpy as np, torch, torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

CFG = __CFG__
print("CFG", CFG, flush=True)
DATA = os.path.dirname(glob.glob("/kaggle/input/**/train.jsonl", recursive=True)[0])
OUT = "/kaggle/working"
dev_ = torch.device("cuda")
rd = lambda n: [json.loads(l) for l in open(f"{DATA}/{n}.jsonl", encoding="utf-8")]
train, dev, test = rd("train"), rd("dev"), rd("test")
K, P_SHUF = CFG["K"], 0.3


def add_siblings(rows, n):
    """Attach up to n most similar id-adjacent claims of the same split (no labels used)."""
    ids = np.array([r["id"] for r in rows]); order = np.argsort(ids, kind="stable")
    B = [{c[i:i + 2] for i in range(len(c) - 1)} for c in (r["claim"] for r in rows)]
    for pos, qi in enumerate(order):
        cand = []
        for pj in range(max(0, pos - 12), min(len(order), pos + 13)):
            qj = order[pj]
            if qj != qi and abs(int(ids[qj]) - int(ids[qi])) <= 20:
                jac = len(B[qi] & B[qj]) / max(1, len(B[qi] | B[qj]))
                if jac >= 0.1:
                    cand.append((jac, qj))
        cand.sort(key=lambda t: (-t[0], t[1]))
        rows[qi]["sibs"] = [rows[qj]["claim"] for _, qj in cand[:n]]


NSIB = CFG.get("sib", 0)
if NSIB:
    for rows_ in (train, dev, test):
        add_siblings(rows_, NSIB)


def with_sibs(r, ctx):
    if not NSIB or not r.get("sibs"):
        return ctx
    return ctx + " [SEP] 相關敘述：" + "；".join(r["sibs"])


def ctx_text(items):
    return " [SEP] ".join(f"{c[0]}：{c[3]}" for c in items)


def examples(rows, seed, order_sr_first=True):
    """rows: dicts with claim/label/rr/gold. S/R: inject gold (best group) into top-K; NEI: retrieved top-K. 30% shuffle."""
    rg = random.Random(seed)
    ex = []
    sr = [r for r in rows if r["label"] != 2]
    nei = [r for r in rows if r["label"] == 2]
    for r in sr:
        lst, groups = r["rr"], r["gold"]
        best = min(groups, key=lambda g: (len({it[3] for it in g}), len(g)))
        by_key = {(c[4], c[1]): c for c in lst}
        top = lst[0][2] if lst else 0.0
        items = []
        for it in best:
            if (it[3], it[1]) in by_key:
                items.append(by_key[(it[3], it[1])]); continue
            if it[2] is None:
                items = None; break
            items.append([it[0], it[1], top, it[2], it[3]])
        if not items:
            continue
        allg = {(it[3], it[1]) for g in groups for it in g}
        fill = [c for c in lst if (c[4], c[1]) not in allg][:max(0, K - len(items))]
        ctx = sorted(items + fill, key=lambda c: -c[2])[:K]
        if rg.random() < P_SHUF:
            rg.shuffle(ctx)
        ex.append((r["claim"], with_sibs(r, ctx_text(ctx)), r["label"]))
    for r in nei:
        ctx = list(r["rr"][:K])
        if ctx:
            if rg.random() < P_SHUF:
                rg.shuffle(ctx)
            ex.append((r["claim"], with_sibs(r, ctx_text(ctx)), 2))
    rg.shuffle(ex)
    return ex


def pairs(rows, view):
    return [(r["claim"], with_sibs(r, ctx_text(r[view][:K]))) for r in rows]


def train_enc(ex, seed):
    torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)
    tok = AutoTokenizer.from_pretrained(CFG["model"])
    model = AutoModelForSequenceClassification.from_pretrained(CFG["model"], num_labels=3, torch_dtype=torch.float32).to(dev_)
    hp = CFG.get("head_perm")
    if hp:                                              # reorder the NLI head (contradiction/neutral/entailment -> supports/refutes/NEI)
        head = model.classifier.out_proj if hasattr(model.classifier, "out_proj") else model.classifier
        with torch.no_grad():
            pm = torch.tensor(hp, device=dev_)
            head.weight.copy_(head.weight[pm].clone()); head.bias.copy_(head.bias[pm].clone())
    if CFG.get("grad_ckpt", True):
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=CFG["lr"], weight_decay=0.01)
    bs, acc, ep = CFG["bs"], CFG["accum"], CFG["epochs"]
    steps = math.ceil(len(ex) / (bs * acc)) * ep
    sch = get_linear_schedule_with_warmup(opt, int(0.06 * steps), steps)
    scaler = torch.amp.GradScaler("cuda")
    rg = random.Random(seed)
    swap = CFG.get("swap", False)
    t0 = time.time()
    for e in range(ep):
        model.train(); order = list(range(len(ex))); rg.shuffle(order); pend = 0; run = 0.0
        for bi, s in enumerate(range(0, len(order), bs)):
            b = [ex[i] for i in order[s:s + bs]]
            a1, a2 = [x[0] for x in b], [x[1] for x in b]
            enc = tok(a2 if swap else a1, a1 if swap else a2, truncation=True, max_length=CFG["max_len"], padding=True, return_tensors="pt").to(dev_)
            lab = torch.tensor([x[2] for x in b], device=dev_)
            with torch.autocast("cuda", dtype=torch.float16):
                lg = model(**enc).logits
            loss = F.cross_entropy(lg.float(), lab, label_smoothing=CFG.get("ls", 0.0)) / acc
            scaler.scale(loss).backward(); pend += 1
            run = 0.98 * run + 0.02 * loss.item() * acc if bi else loss.item() * acc
            if pend == acc or s + bs >= len(order):
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 1.0)
                scaler.step(opt); scaler.update(); sch.step(); opt.zero_grad(set_to_none=True); pend = 0
            if bi % 200 == 0:
                print(f"ep {e} step {bi}/{len(order)//bs} loss {run:.4f} {time.time()-t0:.0f}s", flush=True)
    model.eval()
    return model, tok


@torch.no_grad()
def infer(model, tok, prs):
    swap = CFG.get("swap", False)
    order = np.argsort([len(a) + len(b) for a, b in prs], kind="stable"); out = np.zeros((len(prs), 3), np.float32)
    for s in range(0, len(order), 64):
        ids = order[s:s + 64]
        a1, a2 = [prs[i][0] for i in ids], [prs[i][1] for i in ids]
        enc = tok(a2 if swap else a1, a1 if swap else a2, truncation=True, max_length=CFG["max_len"], padding=True, return_tensors="pt").to(dev_)
        with torch.autocast("cuda", dtype=torch.float16):
            out[ids] = model(**enc).logits.float().cpu().numpy()
    return out


tag = CFG["tag"]
nf = CFG.get("cf", 0)                                   # 0 = train only; 2 = cross-fit with dev halves (contiguous ids)
dev_order = np.argsort([r["id"] for r in dev], kind="stable")
folds = np.array_split(dev_order, nf) if nf else [np.array([], dtype=int)]
base_ex = examples(train, CFG["seed"])
for f, held in enumerate(folds):
    held = set(held.tolist())
    ex = list(base_ex)
    if nf:
        extra = examples([dev[q] for q in range(len(dev)) if q not in held], seed=1000 + f)
        ex = ex + extra; random.Random(77 + f).shuffle(ex)
    print(f"fold {f}: {len(ex)} examples", flush=True)
    model, tok = train_enc(ex, CFG["seed"] + f)
    for split, rows in (("dev", dev), ("test", test)):
        for view in ("rr", "sel"):
            lg = infer(model, tok, pairs(rows, view))
            np.save(f"{OUT}/{tag}_f{f}_{split}_{view}.npy", lg)
            if split == "dev":
                yy = np.array([r["label"] for r in dev]); idx = np.array(sorted(held)) if nf else np.arange(len(dev))
                print(f"fold {f} {view} dev acc (held-out part) {(lg[idx].argmax(1) == yy[idx]).mean():.4f}", flush=True)
    del model; torch.cuda.empty_cache()
print("DONE", flush=True)
'''


def kaggle_views(spec):
    """讀取（或在本機大顯存 GPU 上重新訓練）一個 Kaggle 成員，回傳 {(split, view): log-softmax}；dev 為 OOF（cf=0 時為只用 train 訓練的模型）。"""
    d = KG_DIR / spec["tag"]
    nf = spec.get("cf", 0)
    need = [d / f"{spec['tag']}_f{f}_{s}_{v}.npy" for f in range(max(nf, 1)) for s in ("dev", "test") for v in ("rr", "sel")]
    if not all(p.exists() for p in need):
        if not KG_RUN_LOCAL:
            print(f"  （略過 {spec['tag']}：找不到 {d} 的 logits；請在 Kaggle GPU 上執行 KG_SCRIPT，或設 KG_RUN_LOCAL=True）")
            return None
        if not (KG_DATA / "train.jsonl").exists():
            export_for_kaggle()
        d.mkdir(parents=True, exist_ok=True)
        code = KG_SCRIPT.replace("__CFG__", repr(spec)).replace('glob.glob("/kaggle/input/**/train.jsonl", recursive=True)[0]', repr(str(KG_DATA / "train.jsonl")))
        exec(compile(code.replace('OUT = "/kaggle/working"', f"OUT = {str(d)!r}"), "kaggle_member", "exec"), {"__name__": "kaggle_member"})
    ids = np.array(dev["id"])
    folds = np.array_split(np.argsort(ids, kind="stable"), nf) if nf else [np.arange(len(ids))]
    res = {}
    for v in ("rr", "sel"):
        oof = np.zeros((len(ids), 3), np.float32)
        tests = []
        for f in range(max(nf, 1)):
            oof[folds[f]] = log_softmax(np.load(d / f"{spec['tag']}_f{f}_dev_{v}.npy"))[folds[f]]
            tests.append(log_softmax(np.load(d / f"{spec['tag']}_f{f}_test_{v}.npy")))
        res[("dev", v)], res[("test", v)] = oof, np.mean(tests, axis=0)
    return res


KG = {}
for _s in KG_SPECS:
    _r = kaggle_views(_s)
    if _r is not None:
        KG[_s["tag"]] = _r
        print(f"Kaggle 成員 {_s['tag']}：dev accuracy 重排器視角 {(_r[('dev', 'rr')].argmax(1) == dev_y).mean():.4f}、選擇器視角 {(_r[('dev', 'sel')].argmax(1) == dev_y).mean():.4f}"
              f"（{'OOF，cross-fit' if _s.get('cf') else '只用 train 訓練'}）")

Kaggle 成員 xlmr_xnli_k6：dev accuracy 重排器視角 0.7707、選擇器視角 0.7740（OOF，cross-fit）
Kaggle 成員 e330_sib2_k4：dev accuracy 重排器視角 0.7697、選擇器視角 0.7790（只用 train 訓練）
Kaggle 成員 xlmr_sib2_k4：dev accuracy 重排器視角 0.7657、選擇器視角 0.7673（只用 train 訓練）
Kaggle 成員 e330_sib3_k6：dev accuracy 重排器視角 0.7720、選擇器視角 0.7740（只用 train 訓練）
Kaggle 成員 mdeberta_sib2_k4：dev accuracy 重排器視角 0.7553、選擇器視角 0.7587（只用 train 訓練）


In [23]:
# ===== 9e. v9 集成：逐步加入新成員（每一步都做配對檢定）、NEI 比例對齊、產生提交檔 =====
def nei_share(lg, b):
    return float(((lg + np.array([0, 0, b])).argmax(1) == 2).mean())


def bias_for_share(lg, share):
    lo, hi = -3.0, 8.0
    for _ in range(60):
        mid = (lo + hi) / 2
        lo, hi = (mid, hi) if nei_share(lg, mid) < share else (lo, mid)
    return (lo + hi) / 2


DEV_SHARES = (0.42, 0.46, 0.50)                                    # 部署時 dev 的 NEI 預測比例大約落在這個範圍


def paired_at_shares(lg_a, lg_b, ok=dev_ev_ok):
    """在相同的 dev NEI 預測比例下比較兩個集成（同一批 dev 題目的配對檢定）。"""
    out = []
    for s_ in DEV_SHARES:
        a_ = ((lg_a + np.array([0, 0, bias_for_share(lg_a, s_)])).argmax(1) == dev_y) & ok
        b_ = ((lg_b + np.array([0, 0, bias_for_share(lg_b, s_)])).argmax(1) == dev_y) & ok
        n10, n01 = int((~a_ & b_).sum()), int((a_ & ~b_).sum())
        out.append((b_.mean() - a_.mean(), n10, n01, (n10 - n01) / math.sqrt(max(1, n10 + n01))))
    return out


ALL9 = {m["tag"]: {(s["name"], "rr"): log_softmax(member_logits(m, s, EV[s["name"]])) for s in (dev, test)} for m in MEMBERS}
ALL9.update(LOCAL9)
ALL9.update(KG)
V7_TAGS = [m["tag"] for m in MEMBERS]
STEPS = [("v7（6 成員）", V7_TAGS),
         ("pilot → 完整版 qwen06", [t for t in V7_TAGS if t != "qwen06_pilot"] + ["qwen06"])]
for _t in ["xlmr_xnli_k6", "e330_sib2_k4", "xlmr_sib2_k4", "e330_sib3_k6", "mdeberta_sib2_k4", "qwen06_sib"]:
    if _t in ALL9:
        STEPS.append((f"+ {_t}", STEPS[-1][1] + [_t]))
ens9 = lambda tags, split: np.mean([ALL9[t][(split, "rr")] for t in tags], axis=0)
_v7d = ens9(V7_TAGS, "dev")
_rows9, _prev = [], _v7d
for name, tags in STEPS:
    lg = ens9(tags, "dev")
    r = {"步驟": name, "成員數": len(tags), "dev acc": round(float((lg.argmax(1) == dev_y).mean()), 4)}
    for s_, (d, n10, n01, z) in zip(DEV_SHARES, paired_at_shares(_prev, lg)):
        r[f"比上一步 @NEI{s_}"] = f"{d:+.4f}（+{n10}/-{n01}, z {z:+.1f}）"
    for s_, (d, n10, n01, z) in zip(DEV_SHARES, paired_at_shares(_v7d, lg)):
        r[f"比 v7 @NEI{s_}"] = f"{d:+.4f}（z {z:+.1f}）"
    _rows9.append(r)
    _prev = lg
print("逐步加入新成員（dev CFEVER；每一格是在相同 NEI 預測比例下的配對比較）")
print(pd.DataFrame(_rows9).to_string(index=False))
V9_TAGS = STEPS[-1][1]                                             # 最終集成：全部步驟的成員
dev_logits9, test_logits9 = ens9(V9_TAGS, "dev"), ens9(V9_TAGS, "test")
acc9 = float((dev_logits9.argmax(1) == dev_y).mean())
print(f"\nv9 最終集成（{len(V9_TAGS)} 成員）：dev accuracy = {acc9:.4f}（v7 為 {(_v7d.argmax(1) == dev_y).mean():.4f}）")
print("混淆矩陣（列=真實，欄=預測；supports/refutes/NEI），未加偏差：")
print(pd.crosstab(pd.Series(dev_y, name="true"), pd.Series(dev_logits9.argmax(1), name="pred")))
_g9 = np.where(np.array(test["id"]) <= 31996, "A", np.where(np.array(["最喜歡" in c for c in test["claim"]]), "B", "C"))
# NEI 比例：v9 集成在 public 上的掃描（對齊 v7 偏差 1.25 / 1.5 / 1.75 / 1.875 / 2.0 時的 test NEI 比例）是
# 0.884 / 0.883 / 0.889 / 0.888 / 0.888（2.25 → 0.882）—— 更強的集成在 1.75～2.0 形成平台（v7 的最佳點是 1.5）。最終兩份提交取平台的兩端：1.75 與 2.0。
FINAL_SHARES = (1.75, 2.0)
V9_SUBS = {}
for _vb in (1.25, 1.5, 1.75, 1.875, 2.0, 2.25):
    _share = nei_share(test_logits, _vb)
    _b9 = bias_for_share(test_logits9, _share)
    _pred = (test_logits9 + np.array([0, 0, _b9])).argmax(1)
    _path = OUTPUT_DIR / f"submission_v9_m{len(V9_TAGS)}_share{_vb}{_sfx}.csv"
    make_submission(_pred, _path)                                 # 證據沿用 v7 的選擇器（EV_OUT）
    V9_SUBS[_vb] = _path
    print(f"{'★ 最終提交 ' if _vb in FINAL_SHARES else '  '}{_path.name}：test NEI 比例 {_share:.3f}（= v7 偏差 {_vb}）→ v9 偏差 {_b9:.3f}；各組 NEI 比例",
          {g: round(float((_pred[_g9 == g] == 2).mean()), 3) for g in "ABC"})

逐步加入新成員（dev CFEVER；每一格是在相同 NEI 預測比例下的配對比較）
                步驟  成員數  dev acc            比上一步 @NEI0.42            比上一步 @NEI0.46             比上一步 @NEI0.5   比 v7 @NEI0.42   比 v7 @NEI0.46    比 v7 @NEI0.5
          v7（6 成員）    6   0.7780   +0.0000（+0/-0, z +0.0）   +0.0000（+0/-0, z +0.0）   +0.0000（+0/-0, z +0.0） +0.0000（z +0.0） +0.0000（z +0.0） +0.0000（z +0.0）
pilot → 完整版 qwen06    6   0.7840 +0.0063（+34/-15, z +2.7） +0.0053（+33/-17, z +2.3） +0.0033（+34/-24, z +1.3） +0.0063（z +2.7） +0.0053（z +2.3） +0.0033（z +1.3）
    + xlmr_xnli_k6    7   0.7907 +0.0010（+29/-26, z +0.4） +0.0030（+30/-21, z +1.3） +0.0030（+31/-22, z +1.2） +0.0073（z +2.6） +0.0083（z +3.1） +0.0063（z +2.1）
    + e330_sib2_k4    8   0.7933 +0.0063（+38/-19, z +2.5） +0.0010（+29/-26, z +0.4） +0.0067（+40/-20, z +2.6） +0.0137（z +4.3） +0.0093（z +3.1） +0.0130（z +3.8）
    + xlmr_sib2_k4    9   0.7960 +0.0017（+20/-15, z +0.8） +0.0043（+31/-18, z +1.9） +0.0027（+22/-14, z +1.3） +0.0153（z +4.3） +0.0137（z +3.9） +0.0157（z +4.3）
    + e330_sib3_k6   

  submission_v9_m12_share1.875.csv：test NEI 比例 0.442（= v7 偏差 1.875）→ v9 偏差 2.135；各組 NEI 比例 {'A': 0.539, 'B': 1.0, 'C': 0.017}
★ 最終提交 submission_v9_m12_share2.0.csv：test NEI 比例 0.451（= v7 偏差 2.0）→ v9 偏差 2.269；各組 NEI 比例 {'A': 0.556, 'B': 1.0, 'C': 0.018}
  submission_v9_m12_share2.25.csv：test NEI 比例 0.467（= v7 偏差 2.25）→ v9 偏差 2.513；各組 NEI 比例 {'A': 0.584, 'B': 1.0, 'C': 0.024}


### 9f. 負面結果：在 dev 上有效、但沒有足夠證據的事後調整（未採用）

以下兩個方法在 dev 上看起來有效，但送到 Kaggle 後 public 沒有跟著進步，依採用規則**不採用**：
- **兩種證據視角平均（TTA）**：每個成員分別讀「重排器前 4 句」與「選擇器前 4 句」，log-softmax 取平均。dev accuracy 0.778 → 0.784，
  但進步只出現在 NEI 偏差 0（NEI 預測比例低）時；在部署的 NEI 比例下配對結果是 +27/−24、+21/−22、+26/−26（沒有差別），public 0.881 → 0.877。
- **兄弟 claim 堆疊**：用 dev 訓練 logistic regression，輸入「集成機率 + 兄弟 claim 的預測機率與文字差異」。dev（依 id 連續 5 折 OOF）在相同 NEI 比例下 +0.7～1.4 pt（z ≈ 2），
  但 public 0.879 → 0.876。推測原因：它是在 dev 上擬合的，學到的是 dev 群內的標籤組成（三類各 1/3），test A 組的 NEI 比例較高，規則轉移不過去。
  同樣的「兄弟 claim」資訊改成**放進模型輸入、用 train 訓練**（9b 的 qwen06_sib、9d 的 *_sib 成員）之後，就同時在 dev 與 public 進步。
- （另外試過、沒有增益的成員：第二顆完整版 Qwen3-0.6B（只換 seed，與第一顆的對錯一致率 91%）；在 Kaggle 上完整微調、證據放 6 句的 Erlangshen-330M（與現有 330M 一致率 90%）。）

# Report

Your Kaggle username: 17568160 

Your public leaderboard score: 0.889

Your private leaderboard score:


1. Describe your method for retrieving Wikipedia documents (5%) and report Recall@N on development set (5%).

Answer: **方法：混合式檢索 + 學習式排序（完整 wiki，不是 sample code 只索引訓練頁面的做法）。**
sample code 只索引「訓練集用到的頁面」，但 dev 的證據頁面只有 10.8% 在其中，所以我改為索引完整 wiki（1,187,751 頁），每頁只保留「標題 + 開頭 110 字」當檢索摘要。每個 claim 用三種互補的廉價訊號產生候選頁面，再用學習式排序器重排：

- **標題比對**：claim 中以子字串出現的 wiki 標題（只取長度 2–24 的標題，單字標題平均每題出現 13 個、幾乎全是噪音；dev 約 80% 的證據頁面標題會原封不動出現在 claim 中）。同名頁面最多取 6 頁、每題最多 40 個候選。
- **BM25**：字元 bigram（hashing 2^21 維，k1=1.2、b=0.75），取前 50，前 20 進候選；處理 claim 沒有直接寫出標題的情況。
- **稠密檢索**：`BAAI/bge-small-zh-v1.5`（約 24M 參數）對同一份摘要編碼，取前 50，前 20 進候選；處理改寫與同義詞。
- **學習式排序**：三者的聯集用 HistGradientBoosting（19 個特徵：標題長度與位置、是否被更長的標題包含、BM25 / 稠密檢索的分數與名次、各訊號是否命中等）預測「這一頁是不是標準證據頁面」。只用訓練集訓練，而且訓練集的排序分數是用 3 折交叉預測得到的，避免訓練與推論時分數分佈不同。最後取前 6 頁進入下一階段。

**Recall@N（development set）**：分母是 dev 中有標準證據的 supports / refutes 題（2000 題）；一題算命中，是指前 N 頁涵蓋至少一組「完整的」標準證據組的所有頁面。

| Recall@N | 只用標題比對 | 只用 BM25 | 只用稠密檢索 | RRF 融合（無學習） | **學習式排序（本方法）** |
|---|---|---|---|---|---|
| @1 | 0.5235 | 0.5400 | 0.5235 | 0.6995 | **0.7915** |
| @3 | 0.7405 | 0.6385 | 0.6575 | 0.8290 | **0.8955** |
| @5 | 0.8255 | 0.6725 | 0.6965 | 0.8535 | **0.9175** |
| @6（實際使用） | 0.8385 | 0.6845 | 0.7135 | 0.8570 | **0.9245** |
| @10 | 0.8635 | 0.7175 | 0.7555 | 0.8760 | **0.9375** |
| @20 | 0.9000 | 0.7585 | 0.7975 | 0.9220 | **0.9410** |

三種訊號各自在 @5 只有 0.67–0.83，簡單的 RRF 融合到 0.8535，學習式排序再提高到 0.9175（+6.4 個百分點）。實際送往下一階段的是前 6 頁（Recall@6 = 0.9245）。

**本題使用的模型與使用方式**：只有稠密檢索用到預訓練模型 `BAAI/bge-small-zh-v1.5`（約 24M 參數，Hugging Face 公開 checkpoint），**直接使用、沒有微調**：把每頁「標題 + 開頭 110 字」與每個 claim 編成向量（取 [CLS]、L2 正規化），以內積取前 50 名。標題比對與 BM25 是規則 / 統計方法；學習式排序器是 scikit-learn 的 HistGradientBoosting，只用 CFEVER train 從頭訓練。（v9 另試過以兄弟 claim 的頁面分數當排序特徵，沒有採用，見 Q4(e)。）

2. Describe your method for retrieving evidence (5%) and report Recall@N on development set  (5%).

Answer: **方法：粗篩 → cross-encoder 重排 → 證據句選擇器。**

- **候選句**：取排序後前 6 頁（每頁只看前 60 句，因為標準證據有 95% 在前 50 句內），用字元 bigram 重疊度粗篩到每題 32 句。
- **句子重排器（cross-encoder）**：微調 `hfl/chinese-macbert-base`（約 102M），輸入 `claim [SEP] 頁面標題：句子`（很多句子以「他」「該」開頭，加入標題才有主詞），max length 128，1 epoch、lr 3e-5、batch 16 × 累積 2。正例是標準證據句；負例是同頁其他句子、被檢索到的其他頁面的句子、以及標準證據句超連結所指頁面的句子（最難的負例），每題取 5 個負例（分數最高的 3 個難負例 + 2 個隨機負例）。
- **證據句選擇器（v5 起）**：分類準確率約 0.78，但 CFEVER 只有 0.73，差距來自「前 5 句沒有涵蓋完整的標準證據組」（候選池其實涵蓋約 96% 的題目）。重排器只看「claim + 單一句子」，看不到同頁其他句子、頁面排名、句子位置，所以我在 dev 的 supports / refutes 題上訓練一個 HistGradientBoosting 選擇器，重排前 12 句，目標是「該句是否屬於任一組標準證據」，特徵包含：重排分數、名次、與第 1 名的分差、句子編號、該頁的頁面機率與名次、同頁在前 5 名的句數、與 claim 的字元二元組重疊與新增涵蓋量。選擇器只改變送出的 5 句證據，不改變分類器的輸入。
- **評估方式**：選擇器的 dev 數字是 5 折（以題目為單位）out-of-fold 預測，沒有偷看自己的標準答案；test 則用以全部 dev 訓練的選擇器。
- 送出的證據：預測為 NOT ENOUGH INFO 的題目填 `[]`，其他題填前 5 句。

**Recall@N（development set）**：分母同樣是 2000 題 supports / refutes；一題算命中，是指前 N 句涵蓋至少一組完整的標準證據組。（多句證據的題目要整組都在前 N 句內才算，所以 @1 不可能很高。）

| Recall@N | 句子重排器（MacBERT） | **+ 證據句選擇器（最終，5 折 OOF）** |
|---|---|---|
| @1 | 0.4955 | **0.4970** |
| @3 | 0.8005 | **0.8275** |
| @5（實際送出） | 0.8615 | **0.8780** |

選擇器把 Recall@5 從 0.8615 提高到 0.8780；分組來看，需要 1 句 / 2 句 / 3 句證據的題目，Recall@5 分別是 0.9356 → 0.9442、0.7856 → 0.8129、0.6509 → 0.6864，多句證據的題目進步最多。對照「完美證據」的分類上界：dev 分類準確率用完美證據是 0.803，用我們檢索到的證據是 0.778。

**本題使用的模型與使用方式**：句子重排器是 `hfl/chinese-macbert-base`（約 102M，Hugging Face 公開 checkpoint），加上 2 類分類頭後**只用 CFEVER train 微調**（設定見上）；證據句選擇器是 scikit-learn 的 HistGradientBoosting，只用 CFEVER dev 的 supports / refutes 題訓練（5 折 OOF 評估）。v6 曾用 `hfl/chinese-roberta-wwm-ext`（約 102M，只用 CFEVER train 微調、hard negative mining）做第二個重排器，沒有增益，最終未使用。

3. Describe your classification method (5%) and report Accuracy on development set  (5%).

Answer: **方法：12 個分類器的集成（每個都 < 1B 參數），輸入 claim + 前 4 句證據；log-softmax 等權平均後，再加上 NEI 偏差。**

- **輸入**：`[CLS] claim [SEP] 標題：證據1 [SEP] 標題：證據2 …`，證據是重排器排序的前 4 句（97% 的標準證據組不超過 3 句）。NLI 預訓練過的模型把證據放前面（NLI 的「前提, 假設」對應「證據, claim」），並把分類頭依「蘊含 / 矛盾 / 中立」重排成 supports / refutes / NEI。
- **訓練資料的關鍵**：三類的訓練輸入必須長得跟測試時一樣。三類都用「同一個重排器對訓練題實際重排的結果」建立證據：取重排後前 4 句（30% 機率打亂順序）；supports / refutes 若標準證據不在前 4 句，就把缺的標準證據放進去、擠掉分數最低的非標準句；NOT ENOUGH INFO 直接用重排器挑的前 4 句。（第一版 v0 在這裡犯了錯，見 Q4(a)。）
- **兄弟 claim（v9 的關鍵）**：CFEVER 的 claim 是成群產生的——同一個來源句會被改寫成好幾個 claim（supports / refutes / NEI 混在一起），而且 id 相鄰。有 6 個成員在證據後面再附上 2～3 句「兄弟 claim」：同一個資料切分中 id 相差 ≤ 20、字元二元組 Jaccard ≥ 0.1、最相似的 claim（只用 claim 文字與 id，不用任何標籤）。模型因此可以比較「這一句和兄弟相比多了 / 改了什麼」。
- **12 個成員與 dev accuracy（單一模型，未加偏差）**：

| 成員 | 參數量 | 訓練 | dev accuracy |
|---|---|---|---|
| `hfl/chinese-macbert-base` | 約 102M | 本機 | 0.7137 |
| `hfl/chinese-roberta-wwm-ext` | 約 102M | 本機 | 0.7227 |
| `IDEA-CCNL/Erlangshen-Roberta-110M-NLI` | 約 110M | 本機 | 0.7470 |
| `IDEA-CCNL/Erlangshen-Roberta-330M-NLI`（seed 22 / 42；凍結前 8 層、Adafactor） | 約 326M | 本機 | 0.7663 / 0.7647 |
| `Qwen/Qwen3-0.6B-Base` + LoRA（全部訓練資料、2 epoch） | 約 0.6B | 本機 | 0.7630 |
| `Qwen/Qwen3-0.6B-Base` + LoRA + 兄弟 claim | 約 0.6B | 本機 | **0.7753** |
| `joeddav/xlm-roberta-large-xnli`（完整微調、證據 6 句、train + 一半 dev 交叉訓練） | 約 560M | Kaggle GPU | 0.7707（OOF） |
| `Erlangshen-Roberta-330M-NLI` + 2 句兄弟 claim（完整微調） | 約 326M | Kaggle GPU | 0.7697 |
| `Erlangshen-Roberta-330M-NLI` + 3 句兄弟 claim、證據 6 句（完整微調） | 約 326M | Kaggle GPU | 0.7720 |
| `xlm-roberta-large-xnli` + 2 句兄弟 claim | 約 560M | Kaggle GPU | 0.7657 |
| `MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7` + 2 句兄弟 claim | 約 278M | Kaggle GPU | 0.7553 |
| **12 成員集成** | | | **0.7963** |

（「Kaggle GPU」＝在 Kaggle Notebook 的 16GB GPU 上用同一份程式訓練，見 notebook 9d；本機 RTX 3050 只有 4GB。OOF ＝ dev 依 id 連續切兩半，各半由「沒看過它」的模型預測。）

**本題使用的模型與使用方式**：上表 12 個成員都是 Hugging Face 上的公開 checkpoint，全部**只用 CFEVER 資料微調**（train；標註「cross-fit」的成員另加上一半的 dev），沒有使用任何外部資料集。訓練方式：encoder 類（MacBERT、RoBERTa、Erlangshen、XLM-R、mDeBERTa）加 3 類分類頭做全參數微調（本機的 330M 因顯存凍結 embedding 與前 8 層、用 Adafactor；其餘 AdamW、lr 8e-6–2e-5、2 epoch）；Erlangshen / XLM-R / mDeBERTa 這三種是別人已在公開 NLI 資料上微調過的 checkpoint，所以沿用其 NLI 分類頭並重排成 supports / refutes / NEI；`Qwen/Qwen3-0.6B-Base` 是凍結的 bf16 權重 + LoRA（r = 16，所有投影層）+ 分類頭，lr 1e-4、2 epoch。推論時 12 個成員的 log-softmax 等權平均。

**Accuracy（development set，3000 題、三類各 1000；集成，未加偏差）＝ 0.7963**（v7 的 6 成員集成為 0.7780）；對應的 dev CFEVER ＝ 0.7550（v7 為 0.7370）。最後再對 NOT ENOUGH INFO 加偏差（選法見 Q4(c)；最終兩份提交的偏差使 test 被判成 NEI 的比例為 43.3% / 45.1%）。

4. … Anything that can strengthen your report. (10%)

Answer: **(a) 訓練資料建構的捷徑（v0 → v1）。** 第一版的 NOT ENOUGH INFO 證據永遠是 4 句，而 supports / refutes 是「標準證據 + 隨機 0–3 句干擾」，約一半不足 4 句，而且標準證據永遠排第一。模型學成「滿 4 句 = NEI」，dev 上 43% 的 supports / refutes 被判成 NEI，public 只有 0.716。改成三類都用同一個重排器實際重排的結果後，public 變成 0.833。

**(b) test 的組成。** test 6000 題可以分成三種來源（我從資料觀察到的，不是官方資訊）：A 組（id ≤ 31996，3000 題）與 train / dev 同型；B 組（id > 31996 且含「最喜歡」，1000 題）是虛構的個人偏好句，模型 100% 判為 NEI；C 組（其餘 2000 題）是把某一頁維基的句子直接抄下來（≈ supports）或只改一個數字（≈ refutes），模型的判斷與「是否只改了數字」的一致率約 99%。因此分數的差異幾乎都來自 A 組。

**(c) NEI 偏差：dev 與 test 的差異。** dev 三類各 1/3，NEI 偏差 0–1.0 最好；但 v7 在 public 上的曲線是 0.856 / 0.871 / 0.881 / 0.876 / 0.870（偏差 0.5 / 1.0 / 1.5 / 2.0 / 2.5）。我用 dev 的混淆矩陣做 label-shift 估計（BBSE），A 組的類別比例只比 dev 略偏 NEI（約 0.33 / 0.30 / 0.37）；也試過用 domain classifier 把 dev 重新加權成像 A 組（AUC 只有 0.55，dev 與 A 組在表面特徵上幾乎分不開），重新加權後的 dev 仍然偏好低偏差。結論是 A 組的差異是「條件分佈」上的（模型認為 50/50 的題目在 A 組更常是 NEI），無法從 dev 推得，所以以「test 的 NEI 預測比例」為調整對象：換了集成之後，選偏差讓 test 被判成 NEI 的比例等於 v7 在某個偏差下的比例（只用 test 的預測分佈，不用 test 標籤）。最終 12 成員集成在 public 上的掃描是 0.884 / 0.883 / **0.889** / 0.888 / 0.888 / 0.882（對齊 v7 偏差 1.25 / 1.5 / 1.75 / 1.875 / 2.0 / 2.25）：比 v7 更強的集成，最佳點往「更多 NEI」移動，並在 1.75～2.0 形成平台（三個相鄰的點都在 0.888 以上，比單點的雜訊可信），2.25 開始下降。因此最終兩份提交取平台的兩端：對齊 1.75 與 2.0（A 組被判成 NEI 的比例 52.3% / 55.6%）。這個選擇不只是看 public 的點估計：(i) 以兩份提交之間實際改變的題數換算，public 上 1.5 → 1.75 是 +0.006（約 114 題改變，z ≈ +2.4），2.0 → 2.25 是 −0.006（約 98 題改變，z ≈ −2.6），1.75 與 2.0 之間分不出高下，所以最佳點落在 1.75～2.0 的證據兩側各約 2 個標準差；(ii) dev 上有獨立的機制可以解釋為什麼最佳點會往上移：把 dev 的 NEI 預測比例從 0.46 提高到 0.50 時，v9 新增的 NEI 預測中有 22.5% 真的是 NEI（v7 只有 13.3%），NEI 比例每提高 0.08，dev CFEVER 只掉 0.73 pt（v7 掉 1.03 pt）——更強的集成「多判 NEI」的代價較小，而 test A 組的 NEI 又比 dev 多，所以最佳比例比 v7 高；(iii) 不論 NEI 比例取多少，12 成員集成在 dev 上都比 v7 高 1.4～2.3 pt（比例 0.30～0.58 全部為正），所以「用哪個集成」與「比例選哪裡」是分開的決定。private 取兩份中較高者，兩份放在平台兩端是對「private 的最佳點偏左或偏右」的對沖。

**(d) 逐步加入成員（v9）：每一步都在相同的 dev NEI 預測比例（0.42 / 0.46 / 0.50）下做配對檢定。**

| 步驟 | dev acc | 比 v7（三個比例；配對檢定 z） | public |
|---|---|---|---|
| v7（6 成員） | 0.7780 | — | 0.881 |
| pilot → 完整版 Qwen3-0.6B | 0.7840 | +0.63 / +0.53 / +0.33 pt（z 2.7 / 2.3 / 1.3） | |
| + XLM-R-large-XNLI、+ 330M 兄弟 claim | 0.7933 | +1.37 / +0.93 / +1.30 pt（z 4.3 / 3.1 / 3.8） | |
| + XLM-R 兄弟、+ 330M 兄弟（3 句） | 0.7953 | +1.53 / +1.33 / +1.73 pt（z 4.2 / 3.5 / 4.4） | 0.886 |
| + mDeBERTa 兄弟 | 0.7937 | +1.73 / +1.43 / +1.80 pt（z 4.5 / 3.6 / 4.5） | 0.884 |
| + Qwen3-0.6B 兄弟（最終，12 成員） | 0.7963 | +1.67 / +1.73 / +2.30 pt（z 4.2 / 4.2 / 5.6） | 0.883（NEI 比例對齊 1.75 時 **0.889**） |

（public 欄除最後一列外都是對齊 v7 偏差 1.5 的比例。）三份 v9 集成在 public 上都高於 v7；三者之間的差異（0.883–0.886）只改變幾十題的預測，在 public 的雜訊範圍內；在對齊 1.75 的比例下 12 成員集成 0.889、10 成員 0.885，與 dev 的結論一致，所以最終使用全部 12 個成員。Leave-one-out 顯示貢獻最大的是三個新家族 / 新輸入的成員（拿掉兄弟 claim 的 330M、XLM-R、完整版 Qwen 各掉 0.3–0.7 pt）；同一個模型只換 seed 則幾乎沒有幫助（第二顆完整版 Qwen 與第一顆的對錯一致率 91%，加入後 −0.1 pt；Kaggle 上完整微調的 330M 與原本兩顆 330M 一致率 90%，加入後 ±0）。也就是說，集成需要的是「錯誤不同」的成員：不同的模型家族，或不同的輸入資訊（兄弟 claim）。

**本題提到的模型與使用方式**：本題各實驗使用的模型都已在 Q1–Q3 列出（bge-small-zh 直接使用；MacBERT / RoBERTa / Erlangshen / XLM-R-XNLI / mDeBERTa-XNLI / Qwen3-0.6B 都只用 CFEVER 微調）；另外 (d) 中沒有採用的成員：第二顆 Qwen3-0.6B + LoRA（只換 seed）、Erlangshen-330M-NLI 完整微調加 6 句證據、XLM-R / Qwen 的 cross-fit 版本，訓練方式與 Q3 相同；(e) 中的群集感知選擇器、兄弟 claim 堆疊是 scikit-learn 的 HistGradientBoosting / logistic regression，只用 CFEVER dev 訓練。

**(e) 嘗試過但沒有採用的方向（都用同一批 dev 題目做配對比較，沒有足夠證據就不採用）。**

- 第二跳（沿第一輪前 2 名句子的超連結補充頁面）：evidence Recall@5 0.8615 → 0.8635，沒有超過事先訂的門檻。
- 第二個重排器（hard negative mining）當選擇器特徵：OOF Recall@5 一樣是 0.8780。
- 群集感知的證據選擇器（替選擇器加入兄弟 claim 的重排結果當特徵）：依 id 連續切 5 折的 OOF Recall@5 0.875 → 0.879，但 public 0.877 → 0.878，差異太小，不採用。
- 群集感知的頁面排序（兄弟 claim 的頁面分數當特徵）：dev page Recall@6 0.9245 → 0.9295，但會改變 C 組（幾乎全對）2–6% 題目的第 1 名頁面，風險大於收益，不採用。
- 兩種證據視角平均（每個成員分別讀重排器前 4 句、選擇器前 4 句再平均）：dev accuracy 0.778 → 0.784，但進步只在 NEI 偏差 0 時出現；在部署的 NEI 比例下配對結果 +27/−24、+21/−22、+26/−26，public 0.881 → 0.877，不採用。
- 兄弟 claim 堆疊（用 dev 訓練 logistic regression，輸入集成機率 + 兄弟 claim 的預測與文字差異）：dev OOF +0.4–0.8 pt，但它是在 dev 上擬合的事後調整，與 refutes 偏差同類（dev 有效、test 未必），證據不足，不採用。同樣的兄弟 claim 資訊改成「放進模型輸入、用 train 訓練」之後，dev 與 public 才同時進步。
- 在 dev 上學集成權重、refutes 偏差、dev 校準：見 v8，dev 有效但 public 變差或不穩定。

**(f) 錯誤分析（最終集成，dev NEI 預測比例 0.46）。** supports 中標準證據有進到分類器前 4 句的 834 題，FEVER 正確率 0.893；refutes 的 857 題只有 0.662，其中 229 題被判成 NEI——「看到了矛盾證據卻判成資訊不足」是目前最大的錯誤來源，一部分是為了配合 test A 組的高 NEI 比例而付出的代價。證據沒有進到分類器的 supports / refutes 共 309 題（頁面沒找到 151 題、頁面有找到但句子不在前 4 句 158 題），這些題目幾乎不可能得分，是檢索端剩下的空間。

**(g) 教訓與限制。**
- 與類別比例有關、在 dev 上擬合的參數（NEI / refutes 偏差、校準、堆疊）都不能直接轉移到 test；真正同時在 dev 與 public 進步的，只有「更強、更多樣的基礎模型」。
- 每次改動都用配對檢定判斷，並把 public 的差異換算成雜訊大小（一份提交只改變幾十題時，±0.003 都是雜訊），兩邊證據一致才採用。
- NEI 比例最後是參考 public 決定的，這部分對 private 的泛化沒有保證；集成帶來的提升則可在 dev 上以配對檢定驗證（z 4–5.6）。

5. Generative AI Usage Statement

Answer: **Generative AI 使用聲明。**

- **使用的 Generative AI**：Anthropic 的 **Claude（Sonnet 5.5，v0–v8；Opus 5.5，v9）**。Claude 協助了：整體 pipeline 的架構設計、全部程式碼的撰寫與修改（檢索、重排器、證據句選擇器、分類器與集成、Kaggle GPU 上的訓練程式、提交檔產生與分析）、實驗結果的統計分析（配對檢定、label-shift 估計等）、除錯，以及這份 Report 的撰寫。
- **本人的角色**：所有程式都在本人的電腦（Windows + WSL2，RTX 3050 4GB）與本人的 Kaggle 帳號（Kaggle Notebook GPU，私人 notebook 與私人 dataset）上實際執行；Report 內的所有 dev 數字與 Kaggle public 分數都是實際跑出、提交得到的，Report 內容由本人對照輸出檢查過。各版本的取捨與最終選哪兩份提交由本人決定。
- **使用的預訓練模型**（每個模型參數量皆 < 1B；沒有呼叫任何外部推論 API；訓練只用 CFEVER 提供的 train / dev / wiki 資料）：
  - `BAAI/bge-small-zh-v1.5`（約 24M）：稠密頁面檢索
  - `hfl/chinese-macbert-base`（約 102M）：句子重排器、分類器成員
  - `hfl/chinese-roberta-wwm-ext`（約 102M）：分類器成員（v6 另用作第二個重排器）
  - `IDEA-CCNL/Erlangshen-Roberta-110M-NLI`（約 110M）、`IDEA-CCNL/Erlangshen-Roberta-330M-NLI`（約 326M）：分類器成員
  - `Qwen/Qwen3-0.6B-Base`（約 0.6B）+ LoRA：分類器成員
  - `joeddav/xlm-roberta-large-xnli`（約 560M）、`MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7`（約 278M）：分類器成員
  - 其中 Erlangshen NLI、XLM-R XNLI、mDeBERTa XNLI 是別人已在外部 NLI 資料上微調過的 checkpoint；已向助教確認此類 checkpoint 可用（2026-11-14 email 回覆「可以的，只要參數量 1b 以下即可」；討論區助教回覆「可以開放使用任何 open pre-trained model / checkpoint，但要在 report 的每一題裡面說明在哪一個地方你用了什麼模型，以及使用/訓練方式」，所以 Q1–Q4 每一題都附了「本題使用的模型與使用方式」）。未使用任何超過 1B 的模型；只下載模型權重，沒有下載或使用這些 checkpoint 當初訓練用的資料集。
- **參考資料（連結）**：作業提供的 sample code（資料讀取、評分函式、提交格式）；CFEVER 資料集與論文說明 https://github.com/IKMLab/CFEVER-data ；Hugging Face Transformers 文件 https://huggingface.co/docs/transformers ；PEFT（LoRA）文件 https://huggingface.co/docs/peft ；scikit-learn HistGradientBoosting 文件 https://scikit-learn.org/stable/modules/ensemble.html#histogram-based-gradient-boosting ；Kaggle Notebooks GPU 說明 https://www.kaggle.com/docs/notebooks ；各模型的 model card：https://huggingface.co/BAAI/bge-small-zh-v1.5、https://huggingface.co/hfl/chinese-macbert-base、https://huggingface.co/hfl/chinese-roberta-wwm-ext、https://huggingface.co/IDEA-CCNL/Erlangshen-Roberta-110M-NLI、https://huggingface.co/IDEA-CCNL/Erlangshen-Roberta-330M-NLI、https://huggingface.co/Qwen/Qwen3-0.6B-Base、https://huggingface.co/joeddav/xlm-roberta-large-xnli、https://huggingface.co/MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7。